# 08 — Starting-Price Parsing

## Bounded question

> How is starting price represented in the source, and which values can be parsed reproducibly?

## Source and grain

This notebook studies the starting-price field in the immutable source database:

- **Database:** `data/raw/form_2015-present/form_2015-present/raceform.db`
- **Table:** `data`
- **Runner-record grain:** one physical source row per runner record
- **Data-row predicate:** `rowid <> 1`
- **Physical source lineage:** preserve the original SQLite `rowid`

The exact starting-price column name will be confirmed from the existing source-field profile before any source query assumes it.

## Evidence-first method

The analysis will proceed from the stored values themselves:

1. confirm the exact source column and SQLite storage classes;
2. identify SQL `NULL`, blank, sentinel and other missing-value conventions;
3. profile distinct raw values and runner-record frequencies;
4. classify observed notation families and annotations;
5. test deterministic parsing rules against every distinct current value;
6. inspect ambiguous, malformed, unusual and jurisdiction-specific cases in context;
7. distinguish exact source evidence from derived analytical values;
8. define failure-detecting validation for both supported and deliberately unsupported inputs.

Observations, interpretation and design decisions will remain separate.

## Explicit exclusions

This notebook will not:

- reopen race-distance or carried-weight analysis;
- infer prices that are not present in the source;
- silently repair malformed or ambiguous values;
- assume that every price is fractional before profiling the field;
- treat candidate decimal odds or implied probabilities as independent official evidence;
- design the final staging or target database schema;
- perform wider betting-market, bookmaker-margin or price-movement analysis.

## Expected database consequence

A defensible parsing rule should allow later processing to preserve separately:

- the exact raw starting-price value;
- detected notation family;
- parsed numerator and denominator where supported;
- candidate decimal odds;
- candidate implied probability;
- favourite, joint-favourite or related markers;
- parse status;
- ambiguity and anomaly flags;
- unresolved unfamiliar structures without coercion.

Any decimal odds or implied probability produced here will be deterministic derivations from the stored source expression, not independently verified market prices.

## Stopping rule

Stop once the notebook has established:

1. a reproducible parsing rule for all supported current values;
2. a documented set of known exceptions and ambiguous structures;
3. an explicit preservation mechanism for raw and unresolved values;
4. failure-detecting validation that rejects unsupported future notation rather than guessing;
5. a bounded conclusion about what the source field does and does not justify.

In [1]:
from pathlib import Path
import sqlite3

import pandas as pd

# Resolve the project root from the notebook location.
PROJECT_ROOT = Path.cwd().resolve().parent
DB_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "form_2015-present"
    / "form_2015-present"
    / "raceform.db"
)

SOURCE_TABLE = "data"
DATA_ROW_PREDICATE = "rowid <> 1"

# Fail immediately if the expected immutable source database is unavailable.
assert DB_PATH.exists(), f"Source database not found: {DB_PATH}"

# SQLite URI used throughout the notebook to enforce read-only access.
READ_ONLY_DB_URI = f"file:{DB_PATH}?mode=ro"

print(f"Project root: {PROJECT_ROOT}")
print(f"Source database: {DB_PATH}")
print(f"Source table: {SOURCE_TABLE}")
print(f"Data-row predicate: {DATA_ROW_PREDICATE}")

Project root: /home/rob/Documents/inside-rails-horse-racing
Source database: /home/rob/Documents/inside-rails-horse-racing/data/raw/form_2015-present/form_2015-present/raceform.db
Source table: data
Data-row predicate: rowid <> 1


In [2]:
# Inspect the source table definition before referring to any presumed
# starting-price column name.
with sqlite3.connect(READ_ONLY_DB_URI, uri=True) as connection:
    source_columns = pd.read_sql_query(
        f"PRAGMA table_info({SOURCE_TABLE});",
        connection,
    )

source_columns

,cid,name,type,notnull,dflt_value,pk
0,0,date,NUMERIC,0,None,0
1,1,course,TEXT,0,None,0
2,2,race_id,INTEGER,0,None,0
3,3,off,TEXT,0,None,0
4,4,race_name,TEXT,0,None,0
5,5,type,TEXT,0,None,0
6,6,class,TEXT,0,None,0
7,7,pattern,TEXT,0,None,0
8,8,rating_band,TEXT,0,None,0
9,9,age_band,TEXT,0,None,0


In [3]:
# Profile the physical representation and missing-value conventions of `sp`
# before attempting to classify or parse any odds notation.
with sqlite3.connect(READ_ONLY_DB_URI, uri=True) as connection:
    sp_storage_profile = pd.read_sql_query(
        f"""
        SELECT
            typeof(sp) AS sqlite_storage_class,
            COUNT(*) AS runner_records,
            SUM(CASE WHEN sp IS NULL THEN 1 ELSE 0 END) AS sql_null_rows,
            SUM(CASE WHEN typeof(sp) = 'text' AND sp = '' THEN 1 ELSE 0 END) AS empty_string_rows,
            SUM(
                CASE
                    WHEN typeof(sp) = 'text'
                     AND sp <> ''
                     AND trim(sp) = ''
                    THEN 1 ELSE 0
                END
            ) AS whitespace_only_rows,
            SUM(
                CASE
                    WHEN typeof(sp) = 'text'
                     AND sp <> trim(sp)
                    THEN 1 ELSE 0
                END
            ) AS outer_whitespace_rows,
            COUNT(DISTINCT sp) AS distinct_raw_values
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY typeof(sp)
        ORDER BY runner_records DESC;
        """,
        connection,
    )

sp_storage_profile

,sqlite_storage_class,runner_records,sql_null_rows,empty_string_rows,whitespace_only_rows,outer_whitespace_rows,distinct_raw_values
0,text,1851285,0,9097,0,0,843


In [4]:
# List every distinct raw starting-price value with its runner-record frequency.
# Keep the empty string visible as an explicit labelled value.
with sqlite3.connect(READ_ONLY_DB_URI, uri=True) as connection:
    sp_value_frequencies = pd.read_sql_query(
        f"""
        SELECT
            CASE
                WHEN sp = '' THEN '<EMPTY STRING>'
                ELSE sp
            END AS raw_sp_display,
            sp AS raw_sp,
            COUNT(*) AS runner_records
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY sp
        ORDER BY runner_records DESC, sp;
        """,
        connection,
    )

sp_value_frequencies

,raw_sp_display,raw_sp,runner_records
0,12/1,12/1,72682
1,16/1,16/1,72258
2,14/1,14/1,66930
3,25/1,25/1,64100
4,10/1,10/1,63656
...,...,...,...
838,1/10,1/10,1
839,1/28F,1/28F,1
840,34/100F,34/100F,1
841,4/6,4/6,1


In [5]:
# Classify every distinct raw `sp` value into broad structural families.
# This is descriptive profiling only; it does not yet define the parser.
import re

def classify_sp_structure(raw_sp: str) -> str:
    if raw_sp == "":
        return "empty_string"

    if raw_sp == "F":
        return "standalone_favourite_marker"

    if re.fullmatch(r"\d+/\d+", raw_sp):
        return "fractional_unmarked"

    if re.fullmatch(r"\d+/\d+F", raw_sp):
        return "fractional_favourite"

    if re.fullmatch(r"\d+/\d+J", raw_sp):
        return "fractional_joint_favourite"

    if re.fullmatch(r"\d+/\d+C", raw_sp):
        return "fractional_co_favourite"

    if re.fullmatch(r"\d+(\.\d+)?", raw_sp):
        return "numeric_without_slash"

    if re.fullmatch(r"[A-Za-z]+", raw_sp):
        return "alphabetic_token"

    return "other_structure"


sp_structure_profile = (
    sp_value_frequencies
    .assign(
        structure_family=lambda frame: frame["raw_sp"].map(classify_sp_structure)
    )
    .groupby("structure_family", as_index=False)
    .agg(
        distinct_raw_values=("raw_sp", "nunique"),
        runner_records=("runner_records", "sum"),
    )
    .sort_values(
        ["runner_records", "distinct_raw_values"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

sp_structure_profile

,structure_family,distinct_raw_values,runner_records
0,fractional_unmarked,576,1641359
1,fractional_favourite,173,170656
2,fractional_joint_favourite,65,21757
3,empty_string,1,9097
4,alphabetic_token,3,5734
5,fractional_co_favourite,24,2681
6,standalone_favourite_marker,1,1


In [6]:
# Inspect every non-standard token and representative examples of each
# favourite-related suffix before interpreting their meaning.
special_sp_values = (
    sp_value_frequencies
    .assign(
        structure_family=lambda frame: frame["raw_sp"].map(classify_sp_structure)
    )
    .loc[
        lambda frame: frame["structure_family"].isin(
            [
                "alphabetic_token",
                "standalone_favourite_marker",
                "fractional_favourite",
                "fractional_joint_favourite",
                "fractional_co_favourite",
            ]
        )
    ]
    .sort_values(
        ["structure_family", "runner_records", "raw_sp"],
        ascending=[True, False, True],
    )
    .groupby("structure_family", group_keys=False)
    .head(25)
    .reset_index(drop=True)
)

special_sp_values[
    ["structure_family", "raw_sp_display", "runner_records"]
]

,structure_family,raw_sp_display,runner_records
0,alphabetic_token,EvensF,5553
1,alphabetic_token,Evs,123
2,alphabetic_token,EvsJ,58
3,fractional_co_favourite,4/1C,590
4,fractional_co_favourite,7/2C,446
...,...,...,...
73,fractional_joint_favourite,13/2J,75
74,fractional_joint_favourite,37/10J,75
75,fractional_joint_favourite,14/5J,72
76,fractional_joint_favourite,13/5J,64


In [7]:
# Examine favourite-marker behaviour at provisional-race level.
# This will help determine whether F, J and C encode different kinds
# of favourite status rather than treating the suffix letters as synonyms.
with sqlite3.connect(READ_ONLY_DB_URI, uri=True) as connection:
    marker_race_profile = pd.read_sql_query(
        f"""
        WITH marked AS (
            SELECT
                date,
                course,
                off,
                sp,
                CASE
                    WHEN sp = 'F' OR sp LIKE '%F' THEN 'F'
                    WHEN sp LIKE '%J' THEN 'J'
                    WHEN sp LIKE '%C' THEN 'C'
                    ELSE NULL
                END AS marker
            FROM {SOURCE_TABLE}
            WHERE {DATA_ROW_PREDICATE}
        ),
        race_marker_counts AS (
            SELECT
                date,
                course,
                off,
                marker,
                COUNT(*) AS marked_runners
            FROM marked
            WHERE marker IS NOT NULL
            GROUP BY
                date,
                course,
                off,
                marker
        )
        SELECT
            marker,
            marked_runners,
            COUNT(*) AS provisional_races,
            SUM(marked_runners) AS runner_records
        FROM race_marker_counts
        GROUP BY
            marker,
            marked_runners
        ORDER BY
            marker,
            marked_runners;
        """,
        connection,
    )

marker_race_profile

,marker,marked_runners,provisional_races,runner_records
0,C,1,2,2
1,C,2,7,14
2,C,3,794,2382
3,C,4,62,248
4,C,5,7,35
5,F,1,175894,175894
6,F,2,138,276
7,F,3,12,36
8,F,4,1,4
9,J,1,92,92


In [8]:
# Examine favourite-marker behaviour at provisional-race level.
# This will help determine whether F, J and C encode different kinds
# of favourite status rather than treating the suffix letters as synonyms.
with sqlite3.connect(READ_ONLY_DB_URI, uri=True) as connection:
    marker_race_profile = pd.read_sql_query(
        f"""
        WITH marked AS (
            SELECT
                date,
                course,
                off,
                sp,
                CASE
                    WHEN sp = 'F' OR sp LIKE '%F' THEN 'F'
                    WHEN sp LIKE '%J' THEN 'J'
                    WHEN sp LIKE '%C' THEN 'C'
                    ELSE NULL
                END AS marker
            FROM {SOURCE_TABLE}
            WHERE {DATA_ROW_PREDICATE}
        ),
        race_marker_counts AS (
            SELECT
                date,
                course,
                off,
                marker,
                COUNT(*) AS marked_runners
            FROM marked
            WHERE marker IS NOT NULL
            GROUP BY
                date,
                course,
                off,
                marker
        )
        SELECT
            marker,
            marked_runners,
            COUNT(*) AS provisional_races,
            SUM(marked_runners) AS runner_records
        FROM race_marker_counts
        GROUP BY
            marker,
            marked_runners
        ORDER BY
            marker,
            marked_runners;
        """,
        connection,
    )

marker_race_profile

,marker,marked_runners,provisional_races,runner_records
0,C,1,2,2
1,C,2,7,14
2,C,3,794,2382
3,C,4,62,248
4,C,5,7,35
5,F,1,175894,175894
6,F,2,138,276
7,F,3,12,36
8,F,4,1,4
9,J,1,92,92


In [9]:
# Inspect every provisional race whose favourite-marker pattern is unusual:
# - C with fewer than three marked runners
# - J with other than two marked runners
# - F with more than one marked runner
# - any marked group containing multiple underlying prices
with sqlite3.connect(READ_ONLY_DB_URI, uri=True) as connection:
    unusual_marker_races = pd.read_sql_query(
        f"""
        WITH marked AS (
            SELECT
                rowid AS source_rowid,
                date,
                course,
                off,
                race_name,
                ran,
                num,
                pos,
                horse,
                sp,
                CASE
                    WHEN sp = 'F' OR sp LIKE '%F' THEN 'F'
                    WHEN sp LIKE '%J' THEN 'J'
                    WHEN sp LIKE '%C' THEN 'C'
                    ELSE NULL
                END AS marker,
                CASE
                    WHEN sp = 'F' THEN ''
                    WHEN sp LIKE '%F'
                      OR sp LIKE '%J'
                      OR sp LIKE '%C'
                    THEN substr(sp, 1, length(sp) - 1)
                    ELSE sp
                END AS base_price
            FROM {SOURCE_TABLE}
            WHERE {DATA_ROW_PREDICATE}
        ),
        unusual_groups AS (
            SELECT
                date,
                course,
                off,
                marker
            FROM marked
            WHERE marker IS NOT NULL
            GROUP BY
                date,
                course,
                off,
                marker
            HAVING
                (marker = 'C' AND COUNT(*) < 3)
                OR (marker = 'J' AND COUNT(*) <> 2)
                OR (marker = 'F' AND COUNT(*) > 1)
                OR COUNT(DISTINCT base_price) > 1
        )
        SELECT
            m.source_rowid,
            m.date,
            m.course,
            m.off,
            m.race_name,
            m.ran,
            m.num,
            m.pos,
            m.horse,
            m.sp,
            m.marker,
            m.base_price
        FROM marked AS m
        INNER JOIN unusual_groups AS u
            ON m.date = u.date
           AND m.course = u.course
           AND m.off = u.off
        ORDER BY
            m.date,
            m.course,
            m.off,
            m.num,
            m.source_rowid;
        """,
        connection,
    )

unusual_marker_races

,source_rowid,date,course,off,race_name,ran,num,pos,horse,sp,marker,base_price
0,5928,2015-01-19,Aqueduct (USA),8:48,Jazil Stakes (Dirt),6,,6,Abraham (USA),39/10,NaN,39/10
1,5983,2015-01-19,Aqueduct (USA),8:48,Jazil Stakes (Dirt),6,,1,Micromanage (USA),13/20F,F,13/20
2,5984,2015-01-19,Aqueduct (USA),8:48,Jazil Stakes (Dirt),6,,2,Turco Bravo (CHI),181/10,NaN,181/10
3,5985,2015-01-19,Aqueduct (USA),8:48,Jazil Stakes (Dirt),6,,4,Misconnect (USA),13/20F,F,13/20
4,5986,2015-01-19,Aqueduct (USA),8:48,Jazil Stakes (Dirt),6,,5,Schivarelli (USA),41/10,NaN,41/10
...,...,...,...,...,...,...,...,...,...,...,...,...
2908,1850698,2026-05-26,Lingfield,19:54,Free Race Replays On attheraces.com Fillies Ha...,7,3,6,Thunderous Love (GB),9/2,NaN,9/2
2909,1850699,2026-05-26,Lingfield,19:54,Free Race Replays On attheraces.com Fillies Ha...,7,4,7,Daisy Roots (GB),6/1,NaN,6/1
2910,1850696,2026-05-26,Lingfield,19:54,Free Race Replays On attheraces.com Fillies Ha...,7,6,4,Bella Bisbee (GB),100/30J,J,100/30
2911,1850697,2026-05-26,Lingfield,19:54,Free Race Replays On attheraces.com Fillies Ha...,7,7,5,Welljudged (GB),20/1,NaN,20/1


In [10]:
# Summarise the unusual favourite-marker groups at provisional-race level.
# This separates the number of exceptional races from the full runner rows
# returned for contextual inspection.
with sqlite3.connect(READ_ONLY_DB_URI, uri=True) as connection:
    unusual_marker_group_summary = pd.read_sql_query(
        f"""
        WITH marked AS (
            SELECT
                date,
                course,
                off,
                CASE
                    WHEN sp = 'F' OR sp LIKE '%F' THEN 'F'
                    WHEN sp LIKE '%J' THEN 'J'
                    WHEN sp LIKE '%C' THEN 'C'
                    ELSE NULL
                END AS marker,
                CASE
                    WHEN sp = 'F' THEN ''
                    WHEN sp LIKE '%F'
                      OR sp LIKE '%J'
                      OR sp LIKE '%C'
                    THEN substr(sp, 1, length(sp) - 1)
                    ELSE sp
                END AS base_price
            FROM {SOURCE_TABLE}
            WHERE {DATA_ROW_PREDICATE}
        ),
        race_marker_groups AS (
            SELECT
                date,
                course,
                off,
                marker,
                COUNT(*) AS marked_runners,
                COUNT(DISTINCT base_price) AS distinct_marked_base_prices
            FROM marked
            WHERE marker IS NOT NULL
            GROUP BY
                date,
                course,
                off,
                marker
        )
        SELECT
            marker,
            marked_runners,
            distinct_marked_base_prices,
            COUNT(*) AS provisional_races
        FROM race_marker_groups
        WHERE
            (marker = 'C' AND marked_runners < 3)
            OR (marker = 'J' AND marked_runners <> 2)
            OR (marker = 'F' AND marked_runners > 1)
            OR distinct_marked_base_prices > 1
        GROUP BY
            marker,
            marked_runners,
            distinct_marked_base_prices
        ORDER BY
            marker,
            marked_runners,
            distinct_marked_base_prices;
        """,
        connection,
    )

unusual_marker_group_summary

,marker,marked_runners,distinct_marked_base_prices,provisional_races
0,C,1,1,2
1,C,2,1,7
2,F,2,1,137
3,F,2,2,1
4,F,3,1,12
5,F,4,1,1
6,J,1,1,92
7,J,3,1,3


In [11]:
# Show where each unusual marker pattern occurs.
# Course names retain the source jurisdiction suffix where one is supplied.
with sqlite3.connect(READ_ONLY_DB_URI, uri=True) as connection:
    unusual_marker_locations = pd.read_sql_query(
        f"""
        WITH marked AS (
            SELECT
                date,
                course,
                off,
                CASE
                    WHEN sp = 'F' OR sp LIKE '%F' THEN 'F'
                    WHEN sp LIKE '%J' THEN 'J'
                    WHEN sp LIKE '%C' THEN 'C'
                    ELSE NULL
                END AS marker,
                CASE
                    WHEN sp = 'F' THEN ''
                    WHEN sp LIKE '%F'
                      OR sp LIKE '%J'
                      OR sp LIKE '%C'
                    THEN substr(sp, 1, length(sp) - 1)
                    ELSE sp
                END AS base_price
            FROM {SOURCE_TABLE}
            WHERE {DATA_ROW_PREDICATE}
        ),
        race_marker_groups AS (
            SELECT
                date,
                course,
                off,
                marker,
                COUNT(*) AS marked_runners,
                COUNT(DISTINCT base_price) AS distinct_marked_base_prices
            FROM marked
            WHERE marker IS NOT NULL
            GROUP BY
                date,
                course,
                off,
                marker
        ),
        unusual_groups AS (
            SELECT *
            FROM race_marker_groups
            WHERE
                (marker = 'C' AND marked_runners < 3)
                OR (marker = 'J' AND marked_runners <> 2)
                OR (marker = 'F' AND marked_runners > 1)
                OR distinct_marked_base_prices > 1
        )
        SELECT
            marker,
            marked_runners,
            distinct_marked_base_prices,
            course,
            COUNT(*) AS provisional_races
        FROM unusual_groups
        GROUP BY
            marker,
            marked_runners,
            distinct_marked_base_prices,
            course
        ORDER BY
            marker,
            marked_runners,
            distinct_marked_base_prices,
            provisional_races DESC,
            course;
        """,
        connection,
    )

unusual_marker_locations

,marker,marked_runners,distinct_marked_base_prices,course,provisional_races
0,C,1,1,Chantilly (FR),1
1,C,1,1,Jebel Ali (UAE),1
2,C,2,1,Chelmsford (AW),1
3,C,2,1,Hexham,1
4,C,2,1,Jebel Ali (UAE),1
...,...,...,...,...,...
69,J,1,1,Southwell (AW),1
70,J,1,1,Thirsk,1
71,J,1,1,Wolverhampton (AW),1
72,J,3,1,San Isidro (ARG),2


In [12]:
# Produce one contextual row per unusual favourite-marker group.
# This lets us distinguish complete races, incomplete extracts and genuine
# inconsistencies without reviewing thousands of individual runner rows.
with sqlite3.connect(READ_ONLY_DB_URI, uri=True) as connection:
    unusual_marker_race_details = pd.read_sql_query(
        f"""
        WITH source_rows AS (
            SELECT
                date,
                course,
                off,
                race_name,
                ran,
                sp,
                CASE
                    WHEN sp = 'F' OR sp LIKE '%F' THEN 'F'
                    WHEN sp LIKE '%J' THEN 'J'
                    WHEN sp LIKE '%C' THEN 'C'
                    ELSE NULL
                END AS marker,
                CASE
                    WHEN sp = 'F' THEN ''
                    WHEN sp LIKE '%F'
                      OR sp LIKE '%J'
                      OR sp LIKE '%C'
                    THEN substr(sp, 1, length(sp) - 1)
                    ELSE sp
                END AS base_price
            FROM {SOURCE_TABLE}
            WHERE {DATA_ROW_PREDICATE}
        ),
        race_context AS (
            SELECT
                date,
                course,
                off,
                MAX(race_name) AS race_name,
                MAX(ran) AS declared_ran,
                COUNT(*) AS source_runner_records
            FROM source_rows
            GROUP BY
                date,
                course,
                off
        ),
        marker_groups AS (
            SELECT
                date,
                course,
                off,
                marker,
                COUNT(*) AS marked_runners,
                COUNT(DISTINCT base_price) AS distinct_marked_base_prices,
                GROUP_CONCAT(sp, ', ') AS marked_raw_prices
            FROM source_rows
            WHERE marker IS NOT NULL
            GROUP BY
                date,
                course,
                off,
                marker
        )
        SELECT
            g.date,
            g.course,
            g.off,
            c.race_name,
            c.declared_ran,
            c.source_runner_records,
            c.source_runner_records - c.declared_ran AS row_difference,
            g.marker,
            g.marked_runners,
            g.distinct_marked_base_prices,
            g.marked_raw_prices
        FROM marker_groups AS g
        INNER JOIN race_context AS c
            ON g.date = c.date
           AND g.course = c.course
           AND g.off = c.off
        WHERE
            (g.marker = 'C' AND g.marked_runners < 3)
            OR (g.marker = 'J' AND g.marked_runners <> 2)
            OR (g.marker = 'F' AND g.marked_runners > 1)
            OR g.distinct_marked_base_prices > 1
        ORDER BY
            g.marker,
            g.marked_runners,
            g.distinct_marked_base_prices DESC,
            g.date,
            g.course,
            g.off;
        """,
        connection,
    )

unusual_marker_race_details


,date,course,off,race_name,declared_ran,source_runner_records,row_difference,marker,marked_runners,distinct_marked_base_prices,marked_raw_prices
0,2017-11-03,Jebel Ali (UAE),11:45,Shadwell Handicap (Dirt),12,12,0,C,1,1,11/4C
1,2024-10-17,Chantilly (FR),12:55,Prix de la Vierge de Lorette (Handicap) (3yo+)...,15,15,0,C,1,1,68/10C
2,2015-05-29,Newcastle,3:15,AIS Training Handicap,10,10,0,C,2,1,"5/1C, 5/1C"
3,2016-09-23,Worcester,2:55,7bets4free.com Handicap Chase,6,6,0,C,2,1,"7/2C, 7/2C"
4,2018-08-01,Redcar,5:30,Get So Much More With Racing UK Handicap (Div II),14,14,0,C,2,1,"4/1C, 4/1C"
...,...,...,...,...,...,...,...,...,...,...,...
250,2026-05-11,Catterick,14:30,Racing Welfare Reminiscence Day Apprentice Han...,13,13,0,J,1,1,5/1J
251,2026-05-26,Lingfield,19:54,Free Race Replays On attheraces.com Fillies Ha...,7,7,0,J,1,1,100/30J
252,2017-12-16,San Isidro (ARG),7:15,Gran Premio Felix de Alzaga Unzue (3yo+) (Turf),15,15,0,J,3,1,"4/5J, 4/5J, 4/5J"
253,2019-09-07,Cidade Jardim (BRZ),7:30,Grande Premio Henrique De Toledo Lara (3yo Fi...,9,9,0,J,3,1,"23/10J, 23/10J, 23/10J"


In [13]:
# Profile the numeric fraction beneath every supported-looking price string.
# Remove only the recognised terminal marker and map even-money tokens to 1/1.
# This remains descriptive: no final parser rule is being declared yet.
def extract_candidate_fraction(raw_sp: str):
    if raw_sp == "":
        return None

    if raw_sp in {"Evs", "EvensF", "EvsJ"}:
        return 1, 1

    if raw_sp == "F":
        return None

    base_price = raw_sp[:-1] if raw_sp[-1:] in {"F", "J", "C"} else raw_sp

    match = re.fullmatch(r"(\d+)/(\d+)", base_price)
    if match is None:
        return None

    return int(match.group(1)), int(match.group(2))


fraction_profile = (
    sp_value_frequencies
    .assign(
        candidate_fraction=lambda frame: frame["raw_sp"].map(
            extract_candidate_fraction
        ),
        numerator=lambda frame: frame["candidate_fraction"].map(
            lambda value: value[0] if value is not None else pd.NA
        ),
        denominator=lambda frame: frame["candidate_fraction"].map(
            lambda value: value[1] if value is not None else pd.NA
        ),
    )
)

fraction_numeric_summary = pd.DataFrame(
    {
        "measure": [
            "distinct raw values with candidate fraction",
            "runner records with candidate fraction",
            "distinct raw values without candidate fraction",
            "runner records without candidate fraction",
            "zero numerators",
            "zero denominators",
            "minimum numerator",
            "maximum numerator",
            "minimum denominator",
            "maximum denominator",
        ],
        "value": [
            int(fraction_profile["candidate_fraction"].notna().sum()),
            int(
                fraction_profile.loc[
                    fraction_profile["candidate_fraction"].notna(),
                    "runner_records",
                ].sum()
            ),
            int(fraction_profile["candidate_fraction"].isna().sum()),
            int(
                fraction_profile.loc[
                    fraction_profile["candidate_fraction"].isna(),
                    "runner_records",
                ].sum()
            ),
            int(fraction_profile["numerator"].eq(0).sum()),
            int(fraction_profile["denominator"].eq(0).sum()),
            int(fraction_profile["numerator"].dropna().min()),
            int(fraction_profile["numerator"].dropna().max()),
            int(fraction_profile["denominator"].dropna().min()),
            int(fraction_profile["denominator"].dropna().max()),
        ],
    }
)

fraction_numeric_summary


,measure,value
0,distinct raw values with candidate fraction,841
1,runner records with candidate fraction,1842187
2,distinct raw values without candidate fraction,2
3,runner records without candidate fraction,9098
4,zero numerators,0
5,zero denominators,0
6,minimum numerator,1
7,maximum numerator,1000
8,minimum denominator,1
9,maximum denominator,100


In [14]:
from math import gcd

# Examine the arithmetic form of every candidate fraction.
fraction_arithmetic_profile = (
    fraction_profile
    .loc[lambda frame: frame["candidate_fraction"].notna()]
    .assign(
        greatest_common_divisor=lambda frame: [
            gcd(int(numerator), int(denominator))
            for numerator, denominator in zip(
                frame["numerator"],
                frame["denominator"],
            )
        ],
        price_side=lambda frame: frame.apply(
            lambda row: (
                "odds_on"
                if row["numerator"] < row["denominator"]
                else "even_money"
                if row["numerator"] == row["denominator"]
                else "odds_against"
            ),
            axis=1,
        ),
    )
)

fraction_form_summary = pd.DataFrame(
    {
        "measure": [
            "distinct candidate fractions",
            "fractions already in lowest terms",
            "fractions not in lowest terms",
            "runner records using unreduced fractions",
            "distinct odds-on raw values",
            "odds-on runner records",
            "distinct even-money raw values",
            "even-money runner records",
            "distinct odds-against raw values",
            "odds-against runner records",
            "maximum greatest common divisor",
        ],
        "value": [
            len(fraction_arithmetic_profile),
            int(fraction_arithmetic_profile["greatest_common_divisor"].eq(1).sum()),
            int(fraction_arithmetic_profile["greatest_common_divisor"].gt(1).sum()),
            int(
                fraction_arithmetic_profile.loc[
                    fraction_arithmetic_profile["greatest_common_divisor"].gt(1),
                    "runner_records",
                ].sum()
            ),
            int(fraction_arithmetic_profile["price_side"].eq("odds_on").sum()),
            int(
                fraction_arithmetic_profile.loc[
                    fraction_arithmetic_profile["price_side"].eq("odds_on"),
                    "runner_records",
                ].sum()
            ),
            int(fraction_arithmetic_profile["price_side"].eq("even_money").sum()),
            int(
                fraction_arithmetic_profile.loc[
                    fraction_arithmetic_profile["price_side"].eq("even_money"),
                    "runner_records",
                ].sum()
            ),
            int(fraction_arithmetic_profile["price_side"].eq("odds_against").sum()),
            int(
                fraction_arithmetic_profile.loc[
                    fraction_arithmetic_profile["price_side"].eq("odds_against"),
                    "runner_records",
                ].sum()
            ),
            int(fraction_arithmetic_profile["greatest_common_divisor"].max()),
        ],
    }
)

fraction_form_summary

,measure,value
0,distinct candidate fractions,841
1,fractions already in lowest terms,696
2,fractions not in lowest terms,145
3,runner records using unreduced fractions,77468
4,distinct odds-on raw values,93
5,odds-on runner records,30449
6,distinct even-money raw values,3
7,even-money runner records,5734
8,distinct odds-against raw values,745
9,odds-against runner records,1806004


In [15]:
# Inspect unreduced fractions and test whether multiple raw starting-price
# strings collapse to the same reduced fraction after removing markers.
reduced_fraction_profile = (
    fraction_arithmetic_profile
    .assign(
        reduced_numerator=lambda frame: (
            frame["numerator"] // frame["greatest_common_divisor"]
        ).astype(int),
        reduced_denominator=lambda frame: (
            frame["denominator"] // frame["greatest_common_divisor"]
        ).astype(int),
        reduced_fraction=lambda frame: (
            frame["reduced_numerator"].astype(str)
            + "/"
            + frame["reduced_denominator"].astype(str)
        ),
    )
)

unreduced_fraction_examples = (
    reduced_fraction_profile
    .loc[lambda frame: frame["greatest_common_divisor"].gt(1)]
    .sort_values(
        ["runner_records", "raw_sp"],
        ascending=[False, True],
    )
    [
        [
            "raw_sp",
            "runner_records",
            "numerator",
            "denominator",
            "greatest_common_divisor",
            "reduced_fraction",
        ]
    ]
    .reset_index(drop=True)
)

reduced_fraction_collisions = (
    reduced_fraction_profile
    .groupby("reduced_fraction", as_index=False)
    .agg(
        distinct_raw_values=("raw_sp", "nunique"),
        raw_values=("raw_sp", lambda values: ", ".join(sorted(values))),
        runner_records=("runner_records", "sum"),
    )
    .loc[lambda frame: frame["distinct_raw_values"].gt(1)]
    .sort_values(
        ["distinct_raw_values", "runner_records"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

print(f"Unreduced raw values: {len(unreduced_fraction_examples):,}")
print(f"Reduced-fraction collisions: {len(reduced_fraction_collisions):,}")

display(unreduced_fraction_examples.head(50))
display(reduced_fraction_collisions.head(50))

Unreduced raw values: 145
Reduced-fraction collisions: 114


,raw_sp,runner_records,numerator,denominator,greatest_common_divisor,reduced_fraction
0,100/30,13605,100,30,10,10/3
1,6/4F,7971,6,4,2,3/2
2,100/30F,4171,100,30,10,10/3
3,4/6F,2047,4,6,2,2/3
4,48/10,1977,48,10,2,24/5
5,58/10,1970,58,10,2,29/5
6,54/10,1920,54,10,2,27/5
7,56/10,1907,56,10,2,28/5
8,66/10,1902,66,10,2,33/5
9,42/10,1878,42,10,2,21/5


,reduced_fraction,distinct_raw_values,raw_values,runner_records
0,8/1,4,"8/1, 8/1C, 8/1F, 8/1J",56557
1,7/1,4,"7/1, 7/1C, 7/1F, 7/1J",47056
2,4/1,4,"4/1, 4/1C, 4/1F, 4/1J",46442
3,5/1,4,"5/1, 5/1C, 5/1F, 5/1J",46094
4,9/2,4,"9/2, 9/2C, 9/2F, 9/2J",45575
5,6/1,4,"6/1, 6/1C, 6/1F, 6/1J",44729
6,7/2,4,"7/2, 7/2C, 7/2F, 7/2J",38617
7,11/2,4,"11/2, 11/2C, 11/2F, 11/2J",34544
8,3/1,4,"3/1, 3/1C, 3/1F, 3/1J",32061
9,13/2,4,"13/2, 13/2C, 13/2F, 13/2J",29206


In [16]:
# Separate marker-related duplicates from genuine arithmetic-equivalence collisions.
# `base_price` removes only the recognised favourite suffix while preserving
# the source's original fraction or even-money token.
def extract_base_price(raw_sp: str) -> str | None:
    if raw_sp == "":
        return None

    if raw_sp == "F":
        return None

    if raw_sp in {"Evs", "EvensF", "EvsJ"}:
        return "Evs"

    return raw_sp[:-1] if raw_sp[-1:] in {"F", "J", "C"} else raw_sp


base_price_equivalence = (
    reduced_fraction_profile
    .assign(
        base_price=lambda frame: frame["raw_sp"].map(extract_base_price)
    )
    .groupby("reduced_fraction", as_index=False)
    .agg(
        distinct_base_prices=("base_price", "nunique"),
        base_prices=(
            "base_price",
            lambda values: ", ".join(sorted(set(values.dropna()))),
        ),
        distinct_raw_values=("raw_sp", "nunique"),
        runner_records=("runner_records", "sum"),
    )
    .loc[lambda frame: frame["distinct_base_prices"].gt(1)]
    .sort_values(
        ["distinct_base_prices", "runner_records"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

print(
    "Reduced fractions represented by multiple underlying source prices: "
    f"{len(base_price_equivalence):,}"
)

base_price_equivalence

Reduced fractions represented by multiple underlying source prices: 2


,reduced_fraction,distinct_base_prices,base_prices,distinct_raw_values,runner_records
0,26/5,2,"26/5, 52/10",3,1900
1,29/4,2,"145/20, 29/4",2,61


In [17]:
# Inspect the sole standalone `F` value and every runner in its provisional race.
with sqlite3.connect(READ_ONLY_DB_URI, uri=True) as connection:
    standalone_f_context = pd.read_sql_query(
        f"""
        WITH target_race AS (
            SELECT
                date,
                course,
                off
            FROM {SOURCE_TABLE}
            WHERE {DATA_ROW_PREDICATE}
              AND sp = 'F'
        )
        SELECT
            d.rowid AS source_rowid,
            d.date,
            d.course,
            d.off,
            d.race_name,
            d.type,
            d.ran,
            d.num,
            d.pos,
            d.horse,
            d.sp,
            d.comment
        FROM {SOURCE_TABLE} AS d
        INNER JOIN target_race AS t
            ON d.date = t.date
           AND d.course = t.course
           AND d.off = t.off
        WHERE {DATA_ROW_PREDICATE.replace("rowid", "d.rowid")}
        ORDER BY
            d.num,
            d.rowid;
        """,
        connection,
    )

standalone_f_context

,source_rowid,date,course,off,race_name,type,ran,num,pos,horse,sp,comment
0,1708869,2025-07-20,Del Mar (USA),1:03,Wickerr Stakes () (3yo+) (Turf),Flat,10,1,10,Watsonville (USA),21/1,
1,1708862,2025-07-20,Del Mar (USA),1:03,Wickerr Stakes () (3yo+) (Turf),Flat,10,2,3,Nineeleventurbo (USA),28/1,
2,1708868,2025-07-20,Del Mar (USA),1:03,Wickerr Stakes () (3yo+) (Turf),Flat,10,3,9,Beyond Brilliant (USA),91/10,
3,1708863,2025-07-20,Del Mar (USA),1:03,Wickerr Stakes () (3yo+) (Turf),Flat,10,4,4,Final Boss (USA),94/10,
4,1708861,2025-07-20,Del Mar (USA),1:03,Wickerr Stakes () (3yo+) (Turf),Flat,10,6,2,Zio Jo (USA),141/10,
5,1708865,2025-07-20,Del Mar (USA),1:03,Wickerr Stakes () (3yo+) (Turf),Flat,10,7,6,Bran (FR),19/5,
6,1708867,2025-07-20,Del Mar (USA),1:03,Wickerr Stakes () (3yo+) (Turf),Flat,10,8,8,Air Force Red (USA),178/10,
7,1708860,2025-07-20,Del Mar (USA),1:03,Wickerr Stakes () (3yo+) (Turf),Flat,10,9,1,Almendares (GB),F,
8,1708864,2025-07-20,Del Mar (USA),1:03,Wickerr Stakes () (3yo+) (Turf),Flat,10,10,5,Astronomer (USA),115/10,Broke well - chased leading pair on outer - sh...
9,1708866,2025-07-20,Del Mar (USA),1:03,Wickerr Stakes () (3yo+) (Turf),Flat,10,11,7,Endlessly (USA),4/1,


In [18]:
# Inspect the most unusual observed fractional prices by numerator,
# denominator and arithmetic reduction before defining the parser.
fraction_extremes = (
    fraction_arithmetic_profile
    .sort_values(
        ["numerator", "denominator", "runner_records"],
        ascending=[False, False, True],
    )
    [
        [
            "raw_sp",
            "runner_records",
            "numerator",
            "denominator",
            "greatest_common_divisor",
            "price_side",
        ]
    ]
    .reset_index(drop=True)
)

display(fraction_extremes.head(50))

,raw_sp,runner_records,numerator,denominator,greatest_common_divisor,price_side
0,1000/1,19,1000,1,1,odds_against
1,933/1,22,933,1,1,odds_against
2,885/100,28,885,100,5,odds_against
3,758/100,51,758,100,2,odds_against
4,750/1,51,750,1,1,odds_against
5,705/100,63,705,100,5,odds_against
6,700/1,18,700,1,1,odds_against
7,639/100,15,639,100,1,odds_against
8,604/100,60,604,100,4,odds_against
9,600/1,70,600,1,1,odds_against


In [19]:
# Profile denominator usage across every parseable starting-price value.
# This helps distinguish conventional fractional prices from likely
# converted decimal or tote representations.
denominator_profile = (
    fraction_arithmetic_profile
    .groupby("denominator", as_index=False)
    .agg(
        distinct_raw_values=("raw_sp", "nunique"),
        runner_records=("runner_records", "sum"),
        minimum_numerator=("numerator", "min"),
        maximum_numerator=("numerator", "max"),
    )
    .sort_values(
        ["runner_records", "denominator"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

denominator_profile

,denominator,distinct_raw_values,runner_records,minimum_numerator,maximum_numerator
0,1,276,1309983,1,1000
1,2,30,215369,1,19
2,10,243,131318,1,269
3,4,35,65573,1,37
4,5,60,54124,1,44
5,8,13,21328,1,27
6,30,5,18978,41,100
7,11,11,6726,1,20
8,20,71,4855,1,183
9,6,5,4832,1,22


In [20]:
# Locate unusual denominator families by course.
# This tests whether precision-heavy fractions are concentrated in
# international jurisdictions or also occur in Britain and Ireland.
with sqlite3.connect(READ_ONLY_DB_URI, uri=True) as connection:
    unusual_denominator_locations = pd.read_sql_query(
        f"""
        WITH parsed AS (
            SELECT
                course,
                sp,
                CASE
                    WHEN sp IN ('', 'F') THEN NULL
                    WHEN sp IN ('Evs', 'EvensF', 'EvsJ') THEN 1
                    ELSE CAST(
                        substr(
                            CASE
                                WHEN substr(sp, -1, 1) IN ('F', 'J', 'C')
                                THEN substr(sp, 1, length(sp) - 1)
                                ELSE sp
                            END,
                            instr(
                                CASE
                                    WHEN substr(sp, -1, 1) IN ('F', 'J', 'C')
                                    THEN substr(sp, 1, length(sp) - 1)
                                    ELSE sp
                                END,
                                '/'
                            ) + 1
                        ) AS INTEGER
                    )
                END AS denominator
            FROM {SOURCE_TABLE}
            WHERE {DATA_ROW_PREDICATE}
        )
        SELECT
            denominator,
            course,
            COUNT(*) AS runner_records,
            COUNT(DISTINCT sp) AS distinct_raw_values
        FROM parsed
        WHERE denominator IN (20, 25, 28, 33, 40, 50, 66, 75, 85, 95, 100)
        GROUP BY
            denominator,
            course
        ORDER BY
            denominator,
            runner_records DESC,
            course;
        """,
        connection,
    )

unusual_denominator_locations

,denominator,course,runner_records,distinct_raw_values
0,20,Sha Tin (HK),853,67
1,20,Happy Valley (HK),522,58
2,20,Woodbine (CAN),449,59
3,20,Saratoga (USA),420,52
4,20,Belmont Park (USA),387,51
...,...,...,...,...
488,100,Toulouse (FR),1,1
489,100,Towcester,1,1
490,100,Turffontein Standside,1,1
491,100,Turfway Park,1,1


In [21]:
# Summarise the geographical spread of the precision-heavy denominator
# families without yet assigning a source or conversion rule.
unusual_denominator_summary = (
    unusual_denominator_locations
    .groupby("denominator", as_index=False)
    .agg(
        courses=("course", "nunique"),
        runner_records=("runner_records", "sum"),
        distinct_course_price_combinations=("distinct_raw_values", "sum"),
        leading_courses=(
            "course",
            lambda values: ", ".join(values.head(10)),
        ),
    )
    .sort_values(
        ["runner_records", "denominator"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

unusual_denominator_summary

,denominator,courses,runner_records,distinct_course_price_combinations,leading_courses
0,20,120,4855,1236,"Sha Tin (HK), Happy Valley (HK), Woodbine (CAN..."
1,100,201,1352,447,"Sha Tin (HK), San Siro (ITY), Capannelle (ITY)..."
2,40,114,1084,211,"Wolverhampton (AW), Southwell (AW), Kempton (A..."
3,25,22,29,22,"Capannelle (ITY), Catterick, Chepstow, Killarn..."
4,50,16,26,19,"San Siro (ITY), Bro Park (SWE), Capannelle (IT..."
5,33,6,7,6,"Ffos Las, Hamilton, Lingfield, Market Rasen, W..."
6,85,4,5,4,"San Siro (ITY), Bro Park (SWE), Churchill Down..."
7,75,4,4,4,"Eagle Farm (AUS), Kentucky Downs (USA), Randwi..."
8,95,3,3,3,"Capannelle (ITY), Ellis Park (USA), San Siro (..."
9,66,2,2,2,"Brighton, Leicester"


In [22]:
# Inspect every precision-heavy price recorded at a course without an explicit
# parenthesised foreign-jurisdiction suffix.
#
# This is a contextual review only. A course name without a suffix is not yet
# being treated as definitive evidence of a British or Irish jurisdiction.
with sqlite3.connect(READ_ONLY_DB_URI, uri=True) as connection:
    domestic_looking_precision_prices = pd.read_sql_query(
        f"""
        WITH parsed AS (
            SELECT
                rowid AS source_rowid,
                date,
                course,
                off,
                race_name,
                ran,
                num,
                pos,
                horse,
                sp,
                CASE
                    WHEN sp IN ('', 'F', 'Evs', 'EvensF', 'EvsJ') THEN NULL
                    ELSE CAST(
                        substr(
                            CASE
                                WHEN substr(sp, -1, 1) IN ('F', 'J', 'C')
                                THEN substr(sp, 1, length(sp) - 1)
                                ELSE sp
                            END,
                            instr(
                                CASE
                                    WHEN substr(sp, -1, 1) IN ('F', 'J', 'C')
                                    THEN substr(sp, 1, length(sp) - 1)
                                    ELSE sp
                                END,
                                '/'
                            ) + 1
                        ) AS INTEGER
                    )
                END AS denominator
            FROM {SOURCE_TABLE}
            WHERE {DATA_ROW_PREDICATE}
        )
        SELECT
            source_rowid,
            date,
            course,
            off,
            race_name,
            ran,
            num,
            pos,
            horse,
            sp,
            denominator
        FROM parsed
        WHERE denominator IN (20, 25, 28, 33, 40, 50, 66, 75, 85, 95, 100)
          AND course NOT LIKE '%(%'
        ORDER BY
            denominator,
            course,
            date,
            off,
            num,
            source_rowid;
        """,
        connection,
    )

domestic_looking_precision_prices

,source_rowid,date,course,off,race_name,ran,num,pos,horse,sp,denominator
0,1759875,2025-11-02,Abu Dhabi,16:30,Catalyst Of Speed Handicap (Turf),8,2,2,Sudden Ambush (GB),19/20F,20
1,1781679,2025-12-20,Abu Dhabi,16:30,Race 7 - Desert Lightning Sprint (Maiden) (Turf),8,7,7,Ocean Runner (IRE),29/20F,20
2,1759906,2025-11-02,Aqueduct,16:35,Turnback the Alarm Stakes (3yo+ Fillies & Mare...,5,3,1,Standoutsensation (USA),9/20F,20
3,1759908,2025-11-02,Aqueduct,16:35,Turnback the Alarm Stakes (3yo+ Fillies & Mare...,5,4,3,Indy Label (USA),91/20,20
4,1763007,2025-11-08,Aqueduct,19:39,Hill Prince Stakes (3yo) (Inner Course) (Turf),9,8,6,Mayor Of Midnight (IRE),89/20,20
...,...,...,...,...,...,...,...,...,...,...,...
1233,757543,2019-08-07,Yarmouth,5:45,Matchbook EBF Future Stayers Novice Stakes (Pl...,5,2,4,Sun Cuisine (USA),30/100F,100
1234,1065486,2021-08-29,Yarmouth,2:40,EBF Future Stayers Novice Stakes (Sire/Dam Res...,5,1,1,Find (GB),30/100F,100
1235,1220289,2022-07-26,Yarmouth,2:10,British Stallion Studs EBF Restricted Novice S...,5,1,1,Wahaaj (IRE),30/100F,100
1236,1381743,2023-08-10,Yarmouth,3:20,Quinnbet Fillies Novice Stakes (GBB Race),5,4,1,Karat Karat (IRE),30/100F,100


In [23]:
# Profile the exact underlying price strings used with precision-heavy
# denominators. Remove only favourite suffixes; preserve the stored fraction.
precision_denominator_base_prices = (
    fraction_arithmetic_profile
    .assign(
        base_price=lambda frame: frame["raw_sp"].map(extract_base_price)
    )
    .loc[
        lambda frame: frame["denominator"].isin(
            [20, 25, 28, 33, 40, 50, 66, 75, 85, 95, 100]
        )
    ]
    .groupby(
        ["denominator", "base_price"],
        as_index=False,
    )
    .agg(
        distinct_raw_values=("raw_sp", "nunique"),
        raw_values=("raw_sp", lambda values: ", ".join(sorted(values))),
        runner_records=("runner_records", "sum"),
        greatest_common_divisor=("greatest_common_divisor", "first"),
    )
    .sort_values(
        ["denominator", "runner_records", "base_price"],
        ascending=[True, False, True],
    )
    .reset_index(drop=True)
)

precision_denominator_base_prices

,denominator,base_price,distinct_raw_values,raw_values,runner_records,greatest_common_divisor
0,20,29/20,2,"29/20, 29/20F",180,1
1,20,23/20,2,"23/20, 23/20F",176,1
2,20,27/20,2,"27/20, 27/20F",164,1
3,20,51/20,2,"51/20, 51/20F",159,1
4,20,17/20,1,17/20F,155,1
...,...,...,...,...,...,...
98,100,279/100,1,279/100F,3,1
99,100,176/100,1,176/100,2,4
100,100,188/100,1,188/100,2,4
101,100,284/100,1,284/100F,2,4


In [24]:
# Profile precision-heavy denominator use by calendar year.
# This tests whether the notation is stable throughout the source or reflects
# changes in the upstream international-price feed or formatting process.
with sqlite3.connect(READ_ONLY_DB_URI, uri=True) as connection:
    precision_denominator_by_year = pd.read_sql_query(
        f"""
        WITH parsed AS (
            SELECT
                substr(CAST(date AS TEXT), 1, 4) AS race_year,
                sp,
                CASE
                    WHEN sp IN ('', 'F', 'Evs', 'EvensF', 'EvsJ') THEN NULL
                    ELSE CAST(
                        substr(
                            CASE
                                WHEN substr(sp, -1, 1) IN ('F', 'J', 'C')
                                THEN substr(sp, 1, length(sp) - 1)
                                ELSE sp
                            END,
                            instr(
                                CASE
                                    WHEN substr(sp, -1, 1) IN ('F', 'J', 'C')
                                    THEN substr(sp, 1, length(sp) - 1)
                                    ELSE sp
                                END,
                                '/'
                            ) + 1
                        ) AS INTEGER
                    )
                END AS denominator
            FROM {SOURCE_TABLE}
            WHERE {DATA_ROW_PREDICATE}
        )
        SELECT
            race_year,
            denominator,
            COUNT(*) AS runner_records,
            COUNT(DISTINCT sp) AS distinct_raw_values
        FROM parsed
        WHERE denominator IN (20, 25, 28, 33, 40, 50, 66, 75, 85, 95, 100)
        GROUP BY
            race_year,
            denominator
        ORDER BY
            race_year,
            denominator;
        """,
        connection,
    )

precision_denominator_by_year

,race_year,denominator,runner_records,distinct_raw_values
0,2015,20,363,63
1,2015,40,19,2
2,2015,50,1,1
3,2015,95,1,1
4,2015,100,130,26
...,...,...,...,...
71,2026,25,1,1
72,2026,40,67,3
73,2026,50,4,3
74,2026,75,1,1


## Observed starting-price representation

### Storage and missingness

The source column is `sp`, declared as SQLite `TEXT`.

Across all 1,851,285 data-like runner records:

- every value is stored as SQLite text;
- there are no SQL `NULL` values;
- there are 9,097 empty strings;
- there are no whitespace-only values;
- there are no values with leading or trailing whitespace;
- there are 843 distinct raw values including the empty string.

### Observed notation families

The current source contains:

- unmarked fractional prices;
- fractional prices suffixed with `F`;
- fractional prices suffixed with `J`;
- fractional prices suffixed with `C`;
- textual even-money forms:
  - `Evs`
  - `EvensF`
  - `EvsJ`
- one standalone `F`;
- no decimal-odds strings;
- no other unclassified structures.

The marker behaviour strongly supports:

- `F` — favourite;
- `J` — joint favourite;
- `C` — co-favourite.

Marker counts within a race are not sufficiently consistent to serve as a parsing-validity rule. The suffix must therefore be preserved literally, with any race-level marker-count anomaly assessed separately.

### Fractional price structure

There are 841 distinct raw values containing reproducible price information, covering 1,842,187 runner records.

Observed numeric components range from:

- numerator: 1 to 1,000;
- denominator: 1 to 100.

No observed fraction has a zero numerator or denominator.

The source does not consistently store fractions in lowest terms:

- 145 distinct raw values are unreduced;
- these cover 77,468 runner records.

Examples include:

- `100/30`;
- `52/10`;
- `145/20`;
- `30/100F`.

Only two reduced price values are represented by more than one underlying source fraction:

- `26/5` and `52/10`;
- `29/4` and `145/20`.

Therefore, the exact stored numerator and denominator are source evidence and must not be overwritten by a reduced representation.

### International and precision-heavy notation

The source contains many denominator forms uncommon in traditional British fractional betting, including:

- 20;
- 25;
- 28;
- 33;
- 40;
- 50;
- 66;
- 75;
- 85;
- 95;
- 100.

These occur extensively at international courses and also occasionally at British and Irish courses.

The notation is present throughout the full 2015–2026 source period.

This is consistent with upstream normalisation of decimal, tote or other returned-price representations into fractional-looking strings, but the source does not provide enough provenance to identify the original price type or conversion rule.

The parser may interpret the stored fraction arithmetically, but it must not claim that the raw value preserves the original official notation used by the relevant jurisdiction.

### Standalone favourite marker

One runner record stores:

```text
F

In [25]:
# Select a compact external-validation sample spanning the important observed
# notation families. We will look these races up individually before deciding
# what the stored fractions represent upstream.
#
# The sample includes:
# - ordinary British fractional prices;
# - odds-on and even-money prices;
# - favourite, joint-favourite and co-favourite markers;
# - unreduced fractions;
# - precision-heavy international fractions;
# - extreme long prices;
# - the standalone favourite marker already inspected.

external_validation_targets = [
    "Evs",
    "EvensF",
    "EvsJ",
    "6/4F",
    "100/30J",
    "4/1C",
    "30/100F",
    "34/100F",
    "145/20",
    "52/10",
    "885/100",
    "181/10",
    "1000/1",
    "F",
]

placeholders = ", ".join("?" for _ in external_validation_targets)

with sqlite3.connect(READ_ONLY_DB_URI, uri=True) as connection:
    external_validation_sample = pd.read_sql_query(
        f"""
        WITH ranked AS (
            SELECT
                rowid AS source_rowid,
                date,
                course,
                off,
                race_name,
                type,
                ran,
                num,
                pos,
                horse,
                sp,
                ROW_NUMBER() OVER (
                    PARTITION BY sp
                    ORDER BY date, course, off, rowid
                ) AS occurrence_number
            FROM {SOURCE_TABLE}
            WHERE {DATA_ROW_PREDICATE}
              AND sp IN ({placeholders})
        )
        SELECT
            source_rowid,
            date,
            course,
            off,
            race_name,
            type,
            ran,
            num,
            pos,
            horse,
            sp
        FROM ranked
        WHERE occurrence_number = 1
        ORDER BY
            CASE sp
                WHEN 'Evs' THEN 1
                WHEN 'EvensF' THEN 2
                WHEN 'EvsJ' THEN 3
                WHEN '6/4F' THEN 4
                WHEN '100/30J' THEN 5
                WHEN '4/1C' THEN 6
                WHEN '30/100F' THEN 7
                WHEN '34/100F' THEN 8
                WHEN '145/20' THEN 9
                WHEN '52/10' THEN 10
                WHEN '885/100' THEN 11
                WHEN '181/10' THEN 12
                WHEN '1000/1' THEN 13
                WHEN 'F' THEN 14
            END;
        """,
        connection,
        params=external_validation_targets,
    )

external_validation_sample

,source_rowid,date,course,off,race_name,type,ran,num,pos,horse,sp
0,14462,2015-02-14,Gulfstream Park (USA),10:00,Hurricane Bertie Stakes (Fillies & Mares) (Dirt),Flat,5,,2,More Than A Party (USA),Evs
1,433,2015-01-01,Cheltenham,1:20,BetBright Novices Chase (Registered As The Dip...,Chase,7,2,1,Ptit Zig (FR),EvensF
2,26403,2015-03-18,Warwick,2:30,JCB Novices Hurdle,Hurdle,3,4,3,Russian Bolero (GER),EvsJ
3,190,2015-01-01,Southwell (AW),12:50,Ladbrokes Mobile Handicap,Flat,8,4,4,Major Rowan (GB),6/4F
4,19622,2015-03-01,Navan (IRE),4:50,Dunsany Handicap Chase,Chase,9,5,4,White Arm (FR),100/30J
5,8948,2015-01-29,Thurles (IRE),3:35,Thurles Handicap Hurdle,Hurdle,8,5,1,Really Unique (IRE),4/1C
6,1413,2015-01-03,Laurel Park (USA),7:29,What A Summer Stakes (Fillies & Mares) (Dirt),Flat,7,,1,Lady Sabelia (USA),30/100F
7,760743,2019-08-15,Merano (ITY),3:05,Premio EBF Terme di Merano () (Fillies & Mares...,Flat,4,,1,Close Your Eyes (ITY),34/100F
8,606022,2018-09-23,San Siro (ITY),4:30,Premio Federico Tesio (3yo+) (Turf),Flat,5,4,2,Presley (ITY),145/20
9,4822,2015-01-16,Palermo (ARG),8:45,Clasico Buenos Aires (3yo+) (Turf),Flat,8,3,2,Life And Glory (ARG),885/100


## External validation of the stratified starting-price sample

A stratified sample of 13 runner records was checked against independent or
jurisdiction-relevant public sources where retrievable.

| Raw `sp` | Runner and race | External finding | Assessment |
|---|---|---|---|
| `Evs` | More Than A Party — 2015 Hurricane Bertie Stakes, Gulfstream Park | No sufficiently clear independent archived price located | Not independently verified |
| `EvensF` | Ptit Zig — 2015 Dipper Novices' Chase, Cheltenham | Contemporary reporting describes Ptit Zig as the **evens favourite** | Exact agreement |
| `EvsJ` | Russian Bolero — 2015 JCB Novices' Hurdle, Warwick | No sufficiently clear independent archived price located | Not independently verified |
| `6/4F` | Major Rowan — 2015 Southwell handicap | No sufficiently clear independent archived price located | Not independently verified |
| `100/30J` | White Arm — 2015 Dunsany Handicap Chase, Navan | No sufficiently clear independent archived price located | Not independently verified |
| `4/1C` | Really Unique — 2015 Thurles Handicap Hurdle | Independent result sources report **4/1 co-favourite**; one states that Really Unique was one of three 4/1 co-favourites | Exact agreement |
| `30/100F` | Lady Sabelia — 2015 What A Summer Stakes, Laurel Park | Contemporary reports describe Lady Sabelia as **3/10 favourite** or **1/5 favourite**, depending on the reporting source | Arithmetic agreement with 3/10; evidence of variation between quoted pre-race or returned prices |
| `34/100F` | Close Your Eyes — 2019 Premio EBF Terme di Merano | No sufficiently clear jurisdiction-native archived returned price located | Not independently verified |
| `145/20` | Presley — 2018 Premio Federico Tesio, San Siro | No sufficiently clear archived returned price located | Not independently verified |
| `885/100` | Life And Glory — 2015 Clasico Buenos Aires, Palermo | No sufficiently clear archived returned price located | Not independently verified |
| `181/10` | Lundi Gras — 2015 Tampa Bay Downs allowance | Comparable Racing Post international results preserve prices such as `181/10`, but no independent chart for this exact runner was retrieved | Source-format corroboration only |
| `1000/1` | Touch Duel — 2015 Kurofune Sho, Kochi | No historical jurisdiction-native win-odds record was retrieved for this exact runner | Not independently verified |
| `F` | Almendares — 2025 Wickerr Stakes, Del Mar | Independent result reporting gives **5/2 favourite** | Raw marker verified; numeric source price omitted |

### Supported findings

The external sample confirms that:

1. ordinary textual and fractional prices can agree exactly with independent
   reporting;
2. `F` and `C` preserve meaningful favourite status;
3. `30/100F` is a valid unreduced representation of `3/10F`, not a malformed
   string;
4. the same race may be described using different quoted prices by different
   reports, so an external article does not automatically establish the exact
   official returned price;
5. standalone `F` can preserve favourite status while omitting a recoverable
   numeric price;
6. historical international returned prices are often difficult to recover
   independently from public web archives.

### Consequence for the parser

The parser should interpret only what is encoded in the stored string:

- parse positive integer fractional components;
- recognise the textual even-money forms;
- preserve `F`, `J`, and `C` literally;
- accept unreduced fractions;
- preserve raw numerator and denominator;
- leave standalone `F` without a parser-derived price;
- keep external corrections and verifications separate from parser output.

External validation supports the arithmetic interpretation of the field, but it
does not establish a single documented upstream conversion method for every
jurisdiction.

In [26]:
from dataclasses import dataclass
from typing import Optional


@dataclass(frozen=True)
class StartingPriceParse:
    raw_sp: object
    notation_family: str
    numerator: Optional[int]
    denominator: Optional[int]
    favourite_marker: Optional[str]
    parse_status: str
    anomaly_flags: tuple[str, ...]


def parse_starting_price(raw_sp: object) -> StartingPriceParse:
    """
    Parse only information explicitly encoded in the source `sp` value.

    External corrections, including the verified 5/2 price for the standalone
    `F` record, are deliberately excluded from this source-string parser.
    """
    if raw_sp is None:
        return StartingPriceParse(
            raw_sp=raw_sp,
            notation_family="missing",
            numerator=None,
            denominator=None,
            favourite_marker=None,
            parse_status="unresolved",
            anomaly_flags=("sql_null",),
        )

    if not isinstance(raw_sp, str):
        return StartingPriceParse(
            raw_sp=raw_sp,
            notation_family="unfamiliar",
            numerator=None,
            denominator=None,
            favourite_marker=None,
            parse_status="unresolved",
            anomaly_flags=("non_text_value",),
        )

    if raw_sp == "":
        return StartingPriceParse(
            raw_sp=raw_sp,
            notation_family="missing",
            numerator=None,
            denominator=None,
            favourite_marker=None,
            parse_status="missing",
            anomaly_flags=(),
        )

    if raw_sp != raw_sp.strip():
        return StartingPriceParse(
            raw_sp=raw_sp,
            notation_family="unfamiliar",
            numerator=None,
            denominator=None,
            favourite_marker=None,
            parse_status="unresolved",
            anomaly_flags=("outer_whitespace",),
        )

    if raw_sp == "F":
        return StartingPriceParse(
            raw_sp=raw_sp,
            notation_family="favourite_marker_only",
            numerator=None,
            denominator=None,
            favourite_marker="F",
            parse_status="price_absent_from_source_string",
            anomaly_flags=("numeric_price_absent",),
        )

    even_money_values = {
        "Evs": None,
        "EvensF": "F",
        "EvsJ": "J",
    }

    if raw_sp in even_money_values:
        return StartingPriceParse(
            raw_sp=raw_sp,
            notation_family="even_money_text",
            numerator=1,
            denominator=1,
            favourite_marker=even_money_values[raw_sp],
            parse_status="parsed",
            anomaly_flags=(),
        )

    favourite_marker = (
        raw_sp[-1] if raw_sp[-1:] in {"F", "J", "C"} else None
    )
    base_price = raw_sp[:-1] if favourite_marker else raw_sp

    match = re.fullmatch(r"([1-9]\d*)/([1-9]\d*)", base_price)

    if match is None:
        return StartingPriceParse(
            raw_sp=raw_sp,
            notation_family="unfamiliar",
            numerator=None,
            denominator=None,
            favourite_marker=favourite_marker,
            parse_status="unresolved",
            anomaly_flags=("unsupported_structure",),
        )

    numerator = int(match.group(1))
    denominator = int(match.group(2))

    flags = []

    if gcd(numerator, denominator) > 1:
        flags.append("fraction_not_in_lowest_terms")

    return StartingPriceParse(
        raw_sp=raw_sp,
        notation_family="fractional",
        numerator=numerator,
        denominator=denominator,
        favourite_marker=favourite_marker,
        parse_status="parsed",
        anomaly_flags=tuple(flags),
    )


distinct_sp_parse_results = pd.DataFrame(
    [
        {
            **parse_starting_price(raw_sp).__dict__,
            "runner_records": runner_records,
        }
        for raw_sp, runner_records in zip(
            sp_value_frequencies["raw_sp"],
            sp_value_frequencies["runner_records"],
        )
    ]
)

distinct_sp_parse_results

,raw_sp,notation_family,numerator,denominator,favourite_marker,parse_status,anomaly_flags,runner_records
0,12/1,fractional,12.0,1.0,NaN,parsed,(),72682
1,16/1,fractional,16.0,1.0,NaN,parsed,(),72258
2,14/1,fractional,14.0,1.0,NaN,parsed,(),66930
3,25/1,fractional,25.0,1.0,NaN,parsed,(),64100
4,10/1,fractional,10.0,1.0,NaN,parsed,(),63656
...,...,...,...,...,...,...,...,...
838,1/10,fractional,1.0,10.0,NaN,parsed,(),1
839,1/28F,fractional,1.0,28.0,F,parsed,(),1
840,34/100F,fractional,34.0,100.0,F,parsed,"(fraction_not_in_lowest_terms,)",1
841,4/6,fractional,4.0,6.0,NaN,parsed,"(fraction_not_in_lowest_terms,)",1


In [27]:
# Summarise parser coverage across distinct raw values and runner records.
parser_coverage_summary = (
    distinct_sp_parse_results
    .groupby(
        ["parse_status", "notation_family"],
        as_index=False,
        dropna=False,
    )
    .agg(
        distinct_raw_values=("raw_sp", "nunique"),
        runner_records=("runner_records", "sum"),
    )
    .sort_values(
        ["runner_records", "distinct_raw_values"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

parser_coverage_summary

,parse_status,notation_family,distinct_raw_values,runner_records
0,parsed,fractional,838,1836453
1,missing,missing,1,9097
2,parsed,even_money_text,3,5734
3,price_absent_from_source_string,favourite_marker_only,1,1


In [28]:
# Summarise anomaly flags across distinct values and runner records.
# A valid parsed price may still carry a non-fatal flag, such as an
# unreduced fractional representation.
anomaly_flag_summary = (
    distinct_sp_parse_results
    .explode("anomaly_flags")
    .assign(
        anomaly_flag=lambda frame: frame["anomaly_flags"].replace(
            {"": pd.NA}
        )
    )
)

anomaly_flag_summary = pd.concat(
    [
        pd.DataFrame(
            {
                "anomaly_flag": ["no_anomaly_flag"],
                "distinct_raw_values": [
                    int(
                        distinct_sp_parse_results["anomaly_flags"]
                        .map(len)
                        .eq(0)
                        .sum()
                    )
                ],
                "runner_records": [
                    int(
                        distinct_sp_parse_results.loc[
                            distinct_sp_parse_results["anomaly_flags"]
                            .map(len)
                            .eq(0),
                            "runner_records",
                        ].sum()
                    )
                ],
            }
        ),
        (
            anomaly_flag_summary
            .dropna(subset=["anomaly_flag"])
            .groupby("anomaly_flag", as_index=False)
            .agg(
                distinct_raw_values=("raw_sp", "nunique"),
                runner_records=("runner_records", "sum"),
            )
        ),
    ],
    ignore_index=True,
).sort_values(
    ["runner_records", "distinct_raw_values"],
    ascending=[False, False],
).reset_index(drop=True)

anomaly_flag_summary

,anomaly_flag,distinct_raw_values,runner_records
0,no_anomaly_flag,697,1773816
1,fraction_not_in_lowest_terms,145,77468
2,numeric_price_absent,1,1


In [29]:
# Exercise the parser against deliberately unsupported or malformed values.
# These cases are not observed in the current source; they test future failure
# behaviour and protect against silent coercion.

parser_test_cases = [
    None,          # SQL NULL
    3.5,           # non-text value
    " 5/2",        # leading whitespace
    "5/2 ",        # trailing whitespace
    "05/2",        # leading-zero numerator
    "5/02",        # leading-zero denominator
    "0/1",         # zero numerator
    "5/0",         # zero denominator
    "5-2",         # wrong separator
    "5/2JF",       # multiple suffix markers
    "5/2X",        # unknown suffix
    "57kg",        # metric notation
    "3.50",        # decimal odds
    "SP",          # non-price token
    "Evens",       # currently unsupported even-money spelling
    "EVS",         # unsupported capitalisation
]

parser_failure_tests = pd.DataFrame(
    [
        parse_starting_price(value).__dict__
        for value in parser_test_cases
    ]
)

parser_failure_tests

,raw_sp,notation_family,numerator,denominator,favourite_marker,parse_status,anomaly_flags
0,None,missing,None,None,NaN,unresolved,"(sql_null,)"
1,3.5,unfamiliar,None,None,NaN,unresolved,"(non_text_value,)"
2,5/2,unfamiliar,None,None,NaN,unresolved,"(outer_whitespace,)"
3,5/2,unfamiliar,None,None,NaN,unresolved,"(outer_whitespace,)"
4,05/2,unfamiliar,None,None,NaN,unresolved,"(unsupported_structure,)"
5,5/02,unfamiliar,None,None,NaN,unresolved,"(unsupported_structure,)"
6,0/1,unfamiliar,None,None,NaN,unresolved,"(unsupported_structure,)"
7,5/0,unfamiliar,None,None,NaN,unresolved,"(unsupported_structure,)"
8,5-2,unfamiliar,None,None,NaN,unresolved,"(unsupported_structure,)"
9,5/2JF,unfamiliar,None,None,F,unresolved,"(unsupported_structure,)"


In [30]:
# Tighten the parser so favourite markers are returned only for fully supported
# structures. Unsupported strings such as `5/2JF` must remain wholly unresolved.

def parse_starting_price(raw_sp: object) -> StartingPriceParse:
    """
    Parse only information explicitly encoded in the source `sp` value.

    External corrections and enrichments are deliberately excluded from this
    source-string parser.
    """
    if raw_sp is None:
        return StartingPriceParse(
            raw_sp=raw_sp,
            notation_family="missing",
            numerator=None,
            denominator=None,
            favourite_marker=None,
            parse_status="unresolved",
            anomaly_flags=("sql_null",),
        )

    if not isinstance(raw_sp, str):
        return StartingPriceParse(
            raw_sp=raw_sp,
            notation_family="unfamiliar",
            numerator=None,
            denominator=None,
            favourite_marker=None,
            parse_status="unresolved",
            anomaly_flags=("non_text_value",),
        )

    if raw_sp == "":
        return StartingPriceParse(
            raw_sp=raw_sp,
            notation_family="missing",
            numerator=None,
            denominator=None,
            favourite_marker=None,
            parse_status="missing",
            anomaly_flags=(),
        )

    if raw_sp != raw_sp.strip():
        return StartingPriceParse(
            raw_sp=raw_sp,
            notation_family="unfamiliar",
            numerator=None,
            denominator=None,
            favourite_marker=None,
            parse_status="unresolved",
            anomaly_flags=("outer_whitespace",),
        )

    if raw_sp == "F":
        return StartingPriceParse(
            raw_sp=raw_sp,
            notation_family="favourite_marker_only",
            numerator=None,
            denominator=None,
            favourite_marker="F",
            parse_status="price_absent_from_source_string",
            anomaly_flags=("numeric_price_absent",),
        )

    even_money_values = {
        "Evs": None,
        "EvensF": "F",
        "EvsJ": "J",
    }

    if raw_sp in even_money_values:
        return StartingPriceParse(
            raw_sp=raw_sp,
            notation_family="even_money_text",
            numerator=1,
            denominator=1,
            favourite_marker=even_money_values[raw_sp],
            parse_status="parsed",
            anomaly_flags=(),
        )

    match = re.fullmatch(
        r"([1-9]\d*)/([1-9]\d*)([FJC]?)",
        raw_sp,
    )

    if match is None:
        return StartingPriceParse(
            raw_sp=raw_sp,
            notation_family="unfamiliar",
            numerator=None,
            denominator=None,
            favourite_marker=None,
            parse_status="unresolved",
            anomaly_flags=("unsupported_structure",),
        )

    numerator = int(match.group(1))
    denominator = int(match.group(2))
    favourite_marker = match.group(3) or None

    flags = []

    if gcd(numerator, denominator) > 1:
        flags.append("fraction_not_in_lowest_terms")

    return StartingPriceParse(
        raw_sp=raw_sp,
        notation_family="fractional",
        numerator=numerator,
        denominator=denominator,
        favourite_marker=favourite_marker,
        parse_status="parsed",
        anomaly_flags=tuple(flags),
    )


# Re-run the deliberate failure cases after tightening the supported grammar.
parser_failure_tests = pd.DataFrame(
    [
        parse_starting_price(value).__dict__
        for value in parser_test_cases
    ]
)

parser_failure_tests

,raw_sp,notation_family,numerator,denominator,favourite_marker,parse_status,anomaly_flags
0,None,missing,None,None,None,unresolved,"(sql_null,)"
1,3.5,unfamiliar,None,None,None,unresolved,"(non_text_value,)"
2,5/2,unfamiliar,None,None,None,unresolved,"(outer_whitespace,)"
3,5/2,unfamiliar,None,None,None,unresolved,"(outer_whitespace,)"
4,05/2,unfamiliar,None,None,None,unresolved,"(unsupported_structure,)"
5,5/02,unfamiliar,None,None,None,unresolved,"(unsupported_structure,)"
6,0/1,unfamiliar,None,None,None,unresolved,"(unsupported_structure,)"
7,5/0,unfamiliar,None,None,None,unresolved,"(unsupported_structure,)"
8,5-2,unfamiliar,None,None,None,unresolved,"(unsupported_structure,)"
9,5/2JF,unfamiliar,None,None,None,unresolved,"(unsupported_structure,)"


In [31]:
# Re-parse every current distinct source value with the tightened grammar,
# then confirm that all currently observed structures remain accounted for.

distinct_sp_parse_results = pd.DataFrame(
    [
        {
            **parse_starting_price(raw_sp).__dict__,
            "runner_records": runner_records,
        }
        for raw_sp, runner_records in zip(
            sp_value_frequencies["raw_sp"],
            sp_value_frequencies["runner_records"],
        )
    ]
)

current_parser_validation = pd.DataFrame(
    {
        "measure": [
            "distinct raw values",
            "runner records",
            "parsed distinct raw values",
            "parsed runner records",
            "missing distinct raw values",
            "missing runner records",
            "marker-only distinct raw values",
            "marker-only runner records",
            "unsupported current distinct raw values",
            "unsupported current runner records",
        ],
        "value": [
            len(distinct_sp_parse_results),
            int(distinct_sp_parse_results["runner_records"].sum()),
            int(distinct_sp_parse_results["parse_status"].eq("parsed").sum()),
            int(
                distinct_sp_parse_results.loc[
                    distinct_sp_parse_results["parse_status"].eq("parsed"),
                    "runner_records",
                ].sum()
            ),
            int(distinct_sp_parse_results["parse_status"].eq("missing").sum()),
            int(
                distinct_sp_parse_results.loc[
                    distinct_sp_parse_results["parse_status"].eq("missing"),
                    "runner_records",
                ].sum()
            ),
            int(
                distinct_sp_parse_results["parse_status"]
                .eq("price_absent_from_source_string")
                .sum()
            ),
            int(
                distinct_sp_parse_results.loc[
                    distinct_sp_parse_results["parse_status"].eq(
                        "price_absent_from_source_string"
                    ),
                    "runner_records",
                ].sum()
            ),
            int(distinct_sp_parse_results["parse_status"].eq("unresolved").sum()),
            int(
                distinct_sp_parse_results.loc[
                    distinct_sp_parse_results["parse_status"].eq("unresolved"),
                    "runner_records",
                ].sum()
            ),
        ],
    }
)

current_parser_validation

,measure,value
0,distinct raw values,843
1,runner records,1851285
2,parsed distinct raw values,841
3,parsed runner records,1842187
4,missing distinct raw values,1
5,missing runner records,9097
6,marker-only distinct raw values,1
7,marker-only runner records,1
8,unsupported current distinct raw values,0
9,unsupported current runner records,0


In [32]:
# Profile empty starting-price values in race and runner context before deciding
# which records require individual external investigation.
with sqlite3.connect(READ_ONLY_DB_URI, uri=True) as connection:
    missing_sp_profile = pd.read_sql_query(
        f"""
        WITH race_context AS (
            SELECT
                date,
                course,
                off,
                COUNT(*) AS source_runner_records,
                MAX(ran) AS declared_ran,
                SUM(CASE WHEN sp = '' THEN 1 ELSE 0 END) AS blank_sp_rows,
                COUNT(DISTINCT CASE WHEN sp <> '' THEN sp END) AS nonblank_sp_values
            FROM {SOURCE_TABLE}
            WHERE {DATA_ROW_PREDICATE}
            GROUP BY
                date,
                course,
                off
        )
        SELECT
            CASE
                WHEN r.blank_sp_rows = r.source_runner_records
                    THEN 'all_runners_blank'
                WHEN r.blank_sp_rows > 0
                    THEN 'some_runners_blank'
                ELSE 'no_blank_sp'
            END AS race_blank_pattern,
            COUNT(DISTINCT d.date || '|' || d.course || '|' || d.off)
                AS provisional_races,
            COUNT(*) AS blank_runner_records,
            SUM(CASE WHEN d.pos = '' THEN 1 ELSE 0 END) AS blank_pos_rows,
            SUM(CASE WHEN d.pos = '0' THEN 1 ELSE 0 END) AS zero_pos_rows,
            SUM(CASE WHEN d.pos = 'DSQ' THEN 1 ELSE 0 END) AS dsq_rows,
            SUM(
                CASE
                    WHEN d.pos IN (
                        'BD', 'CO', 'F', 'LFT', 'PU',
                        'REF', 'RO', 'RR', 'SU', 'UR'
                    )
                    THEN 1 ELSE 0
                END
            ) AS nonfinish_rows,
            SUM(
                CASE
                    WHEN CAST(d.pos AS TEXT) GLOB '[1-9]*'
                    THEN 1 ELSE 0
                END
            ) AS positive_numeric_pos_rows,
            SUM(
                CASE
                    WHEN r.source_runner_records = r.declared_ran
                    THEN 1 ELSE 0
                END
            ) AS rows_in_complete_extracts,
            SUM(
                CASE
                    WHEN r.source_runner_records < r.declared_ran
                    THEN 1 ELSE 0
                END
            ) AS rows_in_incomplete_extracts
        FROM {SOURCE_TABLE} AS d
        INNER JOIN race_context AS r
            ON d.date = r.date
           AND d.course = r.course
           AND d.off = r.off
        WHERE {DATA_ROW_PREDICATE.replace("rowid", "d.rowid")}
          AND d.sp = ''
        GROUP BY
            race_blank_pattern
        ORDER BY
            blank_runner_records DESC;
        """,
        connection,
    )

missing_sp_profile

,race_blank_pattern,provisional_races,blank_runner_records,blank_pos_rows,zero_pos_rows,dsq_rows,nonfinish_rows,positive_numeric_pos_rows,rows_in_complete_extracts,rows_in_incomplete_extracts
0,all_runners_blank,701,7766,0,0,11,152,7603,7741,25
1,some_runners_blank,194,1331,0,0,1,76,1254,1331,0


In [33]:
# Locate blank starting-price patterns by course and year.
# This tests whether missing prices cluster by jurisdiction, venue, or period.

with sqlite3.connect(READ_ONLY_DB_URI, uri=True) as connection:
    missing_sp_locations = pd.read_sql_query(
        f"""
        WITH race_context AS (
            SELECT
                date,
                course,
                off,
                COUNT(*) AS source_runner_records,
                SUM(CASE WHEN sp = '' THEN 1 ELSE 0 END) AS blank_sp_rows
            FROM {SOURCE_TABLE}
            WHERE {DATA_ROW_PREDICATE}
            GROUP BY
                date,
                course,
                off
        )
        SELECT
            substr(CAST(d.date AS TEXT), 1, 4) AS race_year,
            d.course,
            CASE
                WHEN r.blank_sp_rows = r.source_runner_records
                    THEN 'all_runners_blank'
                ELSE 'some_runners_blank'
            END AS race_blank_pattern,
            COUNT(DISTINCT d.date || '|' || d.course || '|' || d.off)
                AS provisional_races,
            COUNT(*) AS blank_runner_records
        FROM {SOURCE_TABLE} AS d
        INNER JOIN race_context AS r
            ON d.date = r.date
           AND d.course = r.course
           AND d.off = r.off
        WHERE {DATA_ROW_PREDICATE.replace("rowid", "d.rowid")}
          AND d.sp = ''
        GROUP BY
            race_year,
            d.course,
            race_blank_pattern
        ORDER BY
            blank_runner_records DESC,
            provisional_races DESC,
            race_year,
            d.course;
        """,
        connection,
    )

missing_sp_locations

,race_year,course,race_blank_pattern,provisional_races,blank_runner_records
0,2026,Bahrain,all_runners_blank,41,438
1,2024,Bahrain (BHR),all_runners_blank,25,288
2,2025,Bahrain (BHR),all_runners_blank,24,258
3,2022,Sakhir (BHR),all_runners_blank,20,226
4,2026,Abu Dhabi,all_runners_blank,16,224
...,...,...,...,...,...
288,2025,Meydan,some_runners_blank,1,1
289,2025,Newbury,all_runners_blank,1,1
290,2025,Sandown,all_runners_blank,1,1
291,2025,Tokyo (JPN),some_runners_blank,1,1


In [34]:
# Select representative blank-price races from both patterns:
# - the largest all-runners-blank races by venue/year;
# - partial-blank races with the greatest number of missing runner prices.
#
# This gives us concrete cases to inspect before deciding whether blank `sp`
# means unavailable market data, omitted returns, withdrawals, or source error.

with sqlite3.connect(READ_ONLY_DB_URI, uri=True) as connection:
    representative_blank_sp_races = pd.read_sql_query(
        f"""
        WITH race_context AS (
            SELECT
                date,
                course,
                off,
                MAX(race_name) AS race_name,
                MAX(ran) AS declared_ran,
                COUNT(*) AS source_runner_records,
                SUM(CASE WHEN sp = '' THEN 1 ELSE 0 END) AS blank_sp_rows,
                SUM(CASE WHEN sp <> '' THEN 1 ELSE 0 END) AS nonblank_sp_rows
            FROM {SOURCE_TABLE}
            WHERE {DATA_ROW_PREDICATE}
            GROUP BY
                date,
                course,
                off
        ),
        ranked AS (
            SELECT
                *,
                CASE
                    WHEN blank_sp_rows = source_runner_records
                    THEN 'all_runners_blank'
                    ELSE 'some_runners_blank'
                END AS race_blank_pattern,
                ROW_NUMBER() OVER (
                    PARTITION BY
                        CASE
                            WHEN blank_sp_rows = source_runner_records
                            THEN 'all_runners_blank'
                            ELSE 'some_runners_blank'
                        END
                    ORDER BY
                        blank_sp_rows DESC,
                        source_runner_records DESC,
                        date,
                        course,
                        off
                ) AS pattern_rank
            FROM race_context
            WHERE blank_sp_rows > 0
        )
        SELECT
            date,
            course,
            off,
            race_name,
            declared_ran,
            source_runner_records,
            blank_sp_rows,
            nonblank_sp_rows,
            race_blank_pattern
        FROM ranked
        WHERE pattern_rank <= 20
        ORDER BY
            race_blank_pattern,
            pattern_rank;
        """,
        connection,
    )

representative_blank_sp_races

,date,course,off,race_name,declared_ran,source_runner_records,blank_sp_rows,nonblank_sp_rows,race_blank_pattern
0,2025-01-25,Riyadh (KSA),4:35,The Custodian of the Two Holy Mosques Cup (Sau...,20,20,20,0,all_runners_blank
1,2025-01-04,Riyadh (KSA),5:40,King Faisal Cup () (4yo+) (Outer Track) (Dirt),19,19,19,0,all_runners_blank
2,2020-10-11,Pardubice (CZE),3:40,130th Velka Pardubicka se Slavia Pojistovnou (...,18,18,18,0,all_runners_blank
3,2022-01-07,Sakhir (BHR),12:25,Al Wasmiya Cup (Handicap) (4yo+) (Turf),18,18,18,0,all_runners_blank
4,2023-12-08,Bahrain (BHR),11:50,Al Manama Cup by BBK (Handicap) (3yo+) (Outer ...,18,18,18,0,all_runners_blank
5,2024-01-12,Bahrain (BHR),12:15,Al Wasmiya Cup by Al Salam Bank (Local (4yo+) ...,18,18,18,0,all_runners_blank
6,2024-02-02,Bahrain (BHR),12:15,The Hawar Cup (Handicap) (4yo+) (Outer Track) ...,18,18,18,0,all_runners_blank
7,2024-12-20,Bahrain (BHR),4:00,Al Manama Cup By Al Salam Bank (Handicap) (3yo...,18,18,18,0,all_runners_blank
8,2025-01-30,Bahrain (BHR),4:45,Hawar Cup By Cebarco (Handicap) (4yo+) (Inner ...,18,18,18,0,all_runners_blank
9,2026-01-02,Bahrain,16:00,Al Riffa Cup (Handicap) (4yo+) (Outer Track) (...,18,18,18,0,all_runners_blank


In [35]:
# Inspect complete runner-level context for a small sample of races where only
# some runners have a starting price. This tests whether the nonblank values
# follow a systematic pattern such as winner-only, placed-runner-only, or
# selectively available market data.

partial_blank_targets = [
    ("2016-09-04", "Monterrico (PER)", "11:20"),
    ("2015-12-12", "San Isidro (ARG)", "9:15"),
    ("2019-10-13", "Pardubice (CZE)", "3:40"),
    ("2021-06-20", "Durtal (FR)", "4:00"),
    ("2023-10-19", "Deauville (FR)", "3:45"),
]

target_predicates = " OR ".join(
    "(date = ? AND course = ? AND off = ?)"
    for _ in partial_blank_targets
)

query_params = [
    value
    for target in partial_blank_targets
    for value in target
]

with sqlite3.connect(READ_ONLY_DB_URI, uri=True) as connection:
    partial_blank_runner_context = pd.read_sql_query(
        f"""
        SELECT
            rowid AS source_rowid,
            date,
            course,
            off,
            race_name,
            ran,
            num,
            pos,
            horse,
            sp,
            jockey,
            trainer,
            comment
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
          AND ({target_predicates})
        ORDER BY
            date,
            course,
            off,
            CASE
                WHEN CAST(pos AS TEXT) GLOB '[1-9]*'
                THEN CAST(pos AS INTEGER)
                ELSE 999
            END,
            num,
            rowid;
        """,
        connection,
        params=query_params,
    )

partial_blank_runner_context

,source_rowid,date,course,off,race_name,ran,num,pos,horse,sp,jockey,trainer,comment
0,151593,2015-12-12,San Isidro (ARG),9:15,Gran Premio Carlos Pellegrini (3yo+) (Turf),20,9,1,Hi Happy (ARG),29/20,Altair Domingos,P Nickel Filho,
1,151594,2015-12-12,San Isidro (ARG),9:15,Gran Premio Carlos Pellegrini (3yo+) (Turf),20,3,2,Don Inc (ARG),225/10,Adrian M Giannetti,Carlos D Etchechoury,
2,151595,2015-12-12,San Isidro (ARG),9:15,Gran Premio Carlos Pellegrini (3yo+) (Turf),20,5,3,Es Despues (ARG),79/10,Eduardo Ortega Pavon,Jose L Palacios,
3,151596,2015-12-12,San Isidro (ARG),9:15,Gran Premio Carlos Pellegrini (3yo+) (Turf),20,12,4,Must Go On (ARG),232/10,Gustavo E Calvente,Nicolas Martin Ferro,
4,151597,2015-12-12,San Isidro (ARG),9:15,Gran Premio Carlos Pellegrini (3yo+) (Turf),20,2,5,Champion Star (BRZ),83/10,Vagner Leal,J L Aranha,
...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,1416259,2023-10-19,Deauville (FR),3:45,Prix de la Foret dEawy (Handicap) (4yo+) (All-...,16,14,12,Chubasco (FR),,Maxime Guyon,C Lotoux,
86,1416243,2023-10-19,Deauville (FR),3:45,Prix de la Foret dEawy (Handicap) (4yo+) (All-...,16,1,13,Karak (FR),,Benjamin Marie,G Pannier,
87,1416242,2023-10-19,Deauville (FR),3:45,Prix de la Foret dEawy (Handicap) (4yo+) (All-...,16,8,14,Kingsfold (FR),,Aurelien Lemaitre,J-V Toux,
88,1416241,2023-10-19,Deauville (FR),3:45,Prix de la Foret dEawy (Handicap) (4yo+) (All-...,16,13,15,Downeva (FR),,Mlle Lea Bails,Rosine Bouckhuyt,


In [36]:
# Summarise which finishing positions received prices in each sampled
# partial-blank race before checking the races externally.

partial_blank_position_summary = (
    partial_blank_runner_context
    .assign(
        has_price=lambda frame: frame["sp"].ne(""),
        numeric_position=lambda frame: pd.to_numeric(
            frame["pos"],
            errors="coerce",
        ),
    )
    .groupby(
        ["date", "course", "off", "race_name"],
        as_index=False,
    )
    .agg(
        runner_records=("source_rowid", "count"),
        priced_runner_records=("has_price", "sum"),
        blank_runner_records=("has_price", lambda values: (~values).sum()),
        best_position_with_price=(
            "numeric_position",
            lambda values: values[
                partial_blank_runner_context.loc[values.index, "sp"].ne("")
            ].min(),
        ),
        worst_position_with_price=(
            "numeric_position",
            lambda values: values[
                partial_blank_runner_context.loc[values.index, "sp"].ne("")
            ].max(),
        ),
        priced_positions=(
            "numeric_position",
            lambda values: ", ".join(
                str(int(position))
                for position in sorted(
                    values[
                        partial_blank_runner_context.loc[values.index, "sp"].ne("")
                    ].dropna()
                )
            ),
        ),
        priced_raw_values=(
            "sp",
            lambda values: ", ".join(
                value for value in values if value != ""
            ),
        ),
    )
)

partial_blank_position_summary

,date,course,off,race_name,runner_records,priced_runner_records,blank_runner_records,best_position_with_price,worst_position_with_price,priced_positions,priced_raw_values
0,2015-12-12,San Isidro (ARG),9:15,Gran Premio Carlos Pellegrini (3yo+) (Turf),20,5,15,1.0,5.0,"1, 2, 3, 4, 5","29/20, 225/10, 79/10, 232/10, 83/10"
1,2016-09-04,Monterrico (PER),11:20,Premio Polla de Potrillos - Roberto Alvarez Ca...,18,1,17,1.0,1.0,1,7/2
2,2019-10-13,Pardubice (CZE),3:40,129th Velka Pardubicka Se Slavia Pojistovnou (...,20,5,15,1.0,5.0,"1, 2, 3, 4, 5","5/2F, 24/1, 176/1, 7/1, 6/1"
3,2021-06-20,Durtal (FR),4:00,Prix de Thevalles (Conditions) (4yo) (Turf),16,1,15,1.0,1.0,1,39/10
4,2023-10-19,Deauville (FR),3:45,Prix de la Foret dEawy (Handicap) (4yo+) (All-...,16,1,15,1.0,1.0,1,11/2


In [37]:
# Classify every partial-blank race by which finishing positions contain prices.
#
# This tests whether nonblank values systematically belong to:
# - the winner only;
# - a contiguous leading group such as positions 1–5;
# - non-contiguous finishers;
# - runners with non-numeric outcomes.
#
# External evidence from the sampled Deauville race shows that a winner-only
# value may be a converted tote win dividend rather than an ordinary SP.

with sqlite3.connect(READ_ONLY_DB_URI, uri=True) as connection:
    partial_blank_position_patterns = pd.read_sql_query(
        f"""
        WITH partial_blank_races AS (
            SELECT
                date,
                course,
                off
            FROM {SOURCE_TABLE}
            WHERE {DATA_ROW_PREDICATE}
            GROUP BY
                date,
                course,
                off
            HAVING
                SUM(CASE WHEN sp = '' THEN 1 ELSE 0 END) > 0
                AND SUM(CASE WHEN sp <> '' THEN 1 ELSE 0 END) > 0
        ),
        priced_positions AS (
            SELECT
                d.date,
                d.course,
                d.off,
                COUNT(*) AS source_runner_records,
                SUM(CASE WHEN d.sp <> '' THEN 1 ELSE 0 END) AS priced_runners,
                MIN(
                    CASE
                        WHEN d.sp <> ''
                         AND CAST(d.pos AS TEXT) GLOB '[1-9]*'
                        THEN CAST(d.pos AS INTEGER)
                    END
                ) AS minimum_priced_position,
                MAX(
                    CASE
                        WHEN d.sp <> ''
                         AND CAST(d.pos AS TEXT) GLOB '[1-9]*'
                        THEN CAST(d.pos AS INTEGER)
                    END
                ) AS maximum_priced_position,
                COUNT(
                    DISTINCT CASE
                        WHEN d.sp <> ''
                         AND CAST(d.pos AS TEXT) GLOB '[1-9]*'
                        THEN CAST(d.pos AS INTEGER)
                    END
                ) AS distinct_numeric_priced_positions,
                SUM(
                    CASE
                        WHEN d.sp <> ''
                         AND NOT (CAST(d.pos AS TEXT) GLOB '[1-9]*')
                        THEN 1 ELSE 0
                    END
                ) AS priced_non_numeric_outcomes
            FROM {SOURCE_TABLE} AS d
            INNER JOIN partial_blank_races AS p
                ON d.date = p.date
               AND d.course = p.course
               AND d.off = p.off
            WHERE {DATA_ROW_PREDICATE.replace("rowid", "d.rowid")}
            GROUP BY
                d.date,
                d.course,
                d.off
        )
        SELECT
            CASE
                WHEN priced_runners = 1
                 AND minimum_priced_position = 1
                 AND maximum_priced_position = 1
                 AND priced_non_numeric_outcomes = 0
                    THEN 'winner_only'

                WHEN minimum_priced_position = 1
                 AND maximum_priced_position = priced_runners
                 AND distinct_numeric_priced_positions = priced_runners
                 AND priced_non_numeric_outcomes = 0
                    THEN 'contiguous_leading_finishers'

                WHEN priced_non_numeric_outcomes > 0
                    THEN 'includes_non_numeric_outcome'

                ELSE 'other_position_pattern'
            END AS priced_position_pattern,
            priced_runners,
            COUNT(*) AS provisional_races,
            SUM(source_runner_records) AS source_runner_records
        FROM priced_positions
        GROUP BY
            priced_position_pattern,
            priced_runners
        ORDER BY
            provisional_races DESC,
            priced_position_pattern,
            priced_runners;
        """,
        connection,
    )

partial_blank_position_patterns

,priced_position_pattern,priced_runners,provisional_races,source_runner_records
0,winner_only,1,122,1272
1,contiguous_leading_finishers,5,19,200
2,contiguous_leading_finishers,6,4,28
3,contiguous_leading_finishers,9,4,41
4,contiguous_leading_finishers,17,4,73
5,contiguous_leading_finishers,7,3,24
6,contiguous_leading_finishers,8,3,32
7,contiguous_leading_finishers,11,3,37
8,contiguous_leading_finishers,12,3,39
9,other_position_pattern,6,3,21


## Partial starting-price coverage as a retained source behaviour

Among the 194 provisional races containing both blank and nonblank `sp` values,
the placement of nonblank prices is strongly structured.

Observed patterns include:

- 122 races in which only the winner has a nonblank price;
- races in which prices are present for a contiguous group of leading finishers;
- smaller groups with other finishing-position patterns;
- four races in which priced records include a non-numeric finishing outcome.

External inspection of a sampled Deauville winner-only race showed that the
winner's stored fraction corresponded to the published win tote dividend.

This establishes that a blank runner-level `sp` does not always mean an
accidental missing starting price. Depending on the race and jurisdiction, the
field may contain:

- a conventional starting price for every runner;
- a returned win dividend for the winner only;
- returns for a limited set of leading finishers;
- no supplied price or dividend for the race;
- genuinely incomplete or anomalous source coverage.

### Consequence retained for later database design

The eventual database must not collapse all blank `sp` values into one
undifferentiated missing-price category.

It will need to preserve enough race-level context to distinguish, where
supported by evidence:

- complete runner-level price coverage;
- winner-only return coverage;
- leading-finisher return coverage;
- all-runners-blank race-level coverage absence;
- irregular partial coverage;
- unresolved cases requiring manual or external review.

This notebook records the source behaviour and validation evidence only.
The final staging and target-schema implementation remains deferred.

In [38]:
# Identify the partial-blank races whose priced runners follow an irregular
# position pattern, including races with priced non-numeric outcomes.
#
# These are the cases most likely to require contextual or external review
# before we can describe their source semantics defensibly.

with sqlite3.connect(READ_ONLY_DB_URI, uri=True) as connection:
    irregular_partial_price_races = pd.read_sql_query(
        f"""
        WITH partial_blank_races AS (
            SELECT
                date,
                course,
                off
            FROM {SOURCE_TABLE}
            WHERE {DATA_ROW_PREDICATE}
            GROUP BY
                date,
                course,
                off
            HAVING
                SUM(CASE WHEN sp = '' THEN 1 ELSE 0 END) > 0
                AND SUM(CASE WHEN sp <> '' THEN 1 ELSE 0 END) > 0
        ),
        race_context AS (
            SELECT
                d.date,
                d.course,
                d.off,
                MAX(d.race_name) AS race_name,
                MAX(d.ran) AS declared_ran,
                COUNT(*) AS source_runner_records,
                SUM(CASE WHEN d.sp <> '' THEN 1 ELSE 0 END) AS priced_runners,
                MIN(
                    CASE
                        WHEN d.sp <> ''
                         AND CAST(d.pos AS TEXT) GLOB '[1-9]*'
                        THEN CAST(d.pos AS INTEGER)
                    END
                ) AS minimum_priced_position,
                MAX(
                    CASE
                        WHEN d.sp <> ''
                         AND CAST(d.pos AS TEXT) GLOB '[1-9]*'
                        THEN CAST(d.pos AS INTEGER)
                    END
                ) AS maximum_priced_position,
                COUNT(
                    DISTINCT CASE
                        WHEN d.sp <> ''
                         AND CAST(d.pos AS TEXT) GLOB '[1-9]*'
                        THEN CAST(d.pos AS INTEGER)
                    END
                ) AS distinct_numeric_priced_positions,
                SUM(
                    CASE
                        WHEN d.sp <> ''
                         AND NOT (CAST(d.pos AS TEXT) GLOB '[1-9]*')
                        THEN 1 ELSE 0
                    END
                ) AS priced_non_numeric_outcomes,
                GROUP_CONCAT(
                    CASE
                        WHEN d.sp <> ''
                        THEN CAST(d.pos AS TEXT) || ':' || d.sp
                    END,
                    ', '
                ) AS priced_position_values
            FROM {SOURCE_TABLE} AS d
            INNER JOIN partial_blank_races AS p
                ON d.date = p.date
               AND d.course = p.course
               AND d.off = p.off
            WHERE {DATA_ROW_PREDICATE.replace("rowid", "d.rowid")}
            GROUP BY
                d.date,
                d.course,
                d.off
        )
        SELECT
            date,
            course,
            off,
            race_name,
            declared_ran,
            source_runner_records,
            priced_runners,
            priced_non_numeric_outcomes,
            priced_position_values,
            CASE
                WHEN priced_non_numeric_outcomes > 0
                    THEN 'includes_non_numeric_outcome'
                ELSE 'other_position_pattern'
            END AS irregular_pattern
        FROM race_context
        WHERE
            priced_non_numeric_outcomes > 0
            OR NOT (
                minimum_priced_position = 1
                AND maximum_priced_position = priced_runners
                AND distinct_numeric_priced_positions = priced_runners
                AND priced_non_numeric_outcomes = 0
            )
        ORDER BY
            irregular_pattern,
            date,
            course,
            off;
        """,
        connection,
    )

irregular_partial_price_races

,date,course,off,race_name,declared_ran,source_runner_records,priced_runners,priced_non_numeric_outcomes,priced_position_values,irregular_pattern
0,2016-09-14,Kelso,4:15,Barbara McLeod Memorial Maiden Hurdle,6,6,5,1,"F:80/1, 4:28/1, 3:5/2, 2:4/1, 1:5/6F",includes_non_numeric_outcome
1,2018-05-20,Auteuil (FR),5:05,Prix Tanerko (Hurdle) (Handicap) (5yo+) (Turf),15,15,14,3,"4:27/1, 5:16/1, 6:34/1, 7:76/10, 8:9/1, 9:28/1...",includes_non_numeric_outcome
2,2022-11-11,Compiegne (FR),1:50,Prix Lycoming (Hurdle) (Conditions) (3yo Colts...,11,11,10,3,"PU:13/1, PU:72/1, 7:74/1, 6:2/1F, 5:10/1, 4:22...",includes_non_numeric_outcome
3,2023-07-30,Saratoga (USA),10:44,Bowling Green Stakes (4yo+) (Inner Turf),9,9,8,1,"UR:11/20F, 8:59/10, 6:15/2, 5:27/1, 4:33/1, 3:...",includes_non_numeric_outcome
4,2015-06-18,Longchamp (FR),1:20,Prix de lIle Saint-Louis (Handicap) (4yo+) (Turf),15,15,14,0,"15:207/10, 14:229/10, 13:85/1, 12:18/1, 11:156...",other_position_pattern
5,2016-03-03,Chantilly (FR),3:10,Prix du Centre dEntrainement de Chantilly (Han...,14,14,13,0,"14:33/1, 12:114/10, 11:11/5F, 10:131/10, 9:31/...",other_position_pattern
6,2016-04-01,Maisons-Laffitte (FR),2:55,Prix dHerbeville (Claimer) (3yo Fillies) (Stra...,11,11,10,0,"9:9/1, 8:8/1, 7:11/1, 6:37/10F, 4:12/1, 3:4/1,...",other_position_pattern
7,2016-04-24,Chantilly (FR),1:35,Prix de Provence (Maiden) (3yo Colts & Gelding...,11,11,10,0,"11:37/1, 10:51/1, 9:74/10, 8:16/1, 7:42/1, 5:2...",other_position_pattern
8,2016-06-22,Chantilly (FR),2:40,Prix du Terrain des Spahis (Handicap) (4yo) (P...,16,16,15,0,"8:18/1, 12:48/1, 11:21/1, 10:15/1, 6:12/5F, 5:...",other_position_pattern
9,2019-09-14,Woodbine (CAN),7:14,Bold Venture Stakes (3yo+) (All-Weather Track...,7,7,6,0,"1:7/5F, 3:29/20, 5:23/4, 6:114/10, 7:27/1, 2:9...",other_position_pattern


In [39]:
# Inspect only the blank-price runners from the irregular partial-coverage races.
# This should reveal whether the blanks correspond to withdrawals, non-starters,
# special outcomes, or unexplained source omissions.

irregular_race_keys = list(
    irregular_partial_price_races[
        ["date", "course", "off"]
    ].itertuples(index=False, name=None)
)

irregular_predicates = " OR ".join(
    "(d.date = ? AND d.course = ? AND d.off = ?)"
    for _ in irregular_race_keys
)

irregular_params = [
    value
    for race_key in irregular_race_keys
    for value in race_key
]

with sqlite3.connect(READ_ONLY_DB_URI, uri=True) as connection:
    irregular_blank_runner_context = pd.read_sql_query(
        f"""
        SELECT
            d.rowid AS source_rowid,
            d.date,
            d.course,
            d.off,
            d.race_name,
            d.ran,
            d.num,
            d.pos,
            d.horse,
            d.sp,
            d.jockey,
            d.trainer,
            d.comment
        FROM {SOURCE_TABLE} AS d
        WHERE {DATA_ROW_PREDICATE.replace("rowid", "d.rowid")}
          AND d.sp = ''
          AND ({irregular_predicates})
        ORDER BY
            d.date,
            d.course,
            d.off,
            d.num,
            d.rowid;
        """,
        connection,
        params=irregular_params,
    )

irregular_blank_runner_context

,source_rowid,date,course,off,race_name,ran,num,pos,horse,sp,jockey,trainer,comment
0,71825,2015-06-18,Longchamp (FR),1:20,Prix de lIle Saint-Louis (Handicap) (4yo+) (Turf),15,1,8,Iconic (FR),,Vincent Vion,L Nyffels,
1,175236,2016-03-03,Chantilly (FR),3:10,Prix du Centre dEntrainement de Chantilly (Han...,14,14,13,Attawo (FR),,Eddy Hardouin,P Adda,
2,186316,2016-04-01,Maisons-Laffitte (FR),2:55,Prix dHerbeville (Claimer) (3yo Fillies) (Stra...,11,3,5,Karynia (FR),,Jeremie Catineau,C Laffon-Parias,
3,196789,2016-04-24,Chantilly (FR),1:35,Prix de Provence (Maiden) (3yo Colts & Gelding...,11,13,6,Andrea Mantegna (USA),,Thierry Jarnet,A De Royer-Dupre,
4,227188,2016-06-22,Chantilly (FR),2:40,Prix du Terrain des Spahis (Handicap) (4yo) (P...,16,16,9,Charly Green (FR),,Emmanuel Etienne,P Leblanc,
5,266275,2016-09-14,Kelso,4:15,Barbara McLeod Memorial Maiden Hurdle,6,6,F,Kalaharry (IRE),,Sean Quinlan,Iain Jardine,Held up in touch - fell 5th
6,541461,2018-05-20,Auteuil (FR),5:05,Prix Tanerko (Hurdle) (Handicap) (5yo+) (Turf),15,13,RR,Mukonzi Has (FR),,Jerome Claudic,M Rolland,
7,775700,2019-09-14,Woodbine (CAN),7:14,Bold Venture Stakes (3yo+) (All-Weather Track...,7,2,4,Pink Lloyd (CAN),,Eurico Rosa Da Silva,Robert Tiller,
8,826908,2019-12-28,Mahoning Valley (USA),6:10,Allowance Race (Allowance) (2yo Fillies) (Main...,8,5,4,Gagliano (USA),,Luis Antonio Gonzalez,Tina Casalinova,
9,1002909,2021-05-01,Palermo (ARG),10:25,Gran Premio Ciudad de Buenos Aires (3yo+) (Dirt),15,13,12,Storm Rider (ARG),,Juan C Noriega,Nicolas Tartaglia,


## Externally verified special blank-price case

One irregular partial-coverage race contains a blank `sp` for Modern Games,
winner of the 2021 Breeders' Cup Juvenile Turf at Del Mar.

Modern Games was mistakenly removed from the pari-mutuel wagering pools before
the race. Although subsequently permitted to compete, he ran for purse money
only and was not treated as the winner for wagering purposes.

Therefore:

- the blank `sp` is not an accidental missing value;
- Modern Games completed and won the race;
- no runner-level pari-mutuel return applied to him;
- the second-finishing horse was treated as the wagering winner;
- finishing position alone cannot determine whether a price should exist.

### Retained source consequence

Blank starting-price values may represent at least:

- race-level absence of supplied price data;
- winner-only or leading-finisher tote-return coverage;
- a runner excluded from the wagering pool;
- irregular source omissions requiring further review.

The eventual database must preserve runner-level wagering applicability
separately from finishing position and raw price availability where such
evidence can be established.

## Revision to the field interpretation

The source column is named `sp`, but the observed evidence shows that it does not
consist exclusively of conventional runner-level starting prices.

Depending on race and jurisdiction, the field may contain:

- a conventional fixed-odds starting price;
- a fractional representation of a pari-mutuel or tote win dividend;
- returns supplied only for the winner;
- returns supplied for a limited group of leading finishers;
- a favourite-status marker without a numeric source price;
- a blank value because the runner was excluded from wagering;
- a blank value because no race-level price or return was supplied;
- an unexplained source omission.

Therefore, throughout the remainder of this notebook:

- `raw_sp` refers only to the exact stored source value;
- parsed numerator and denominator describe the arithmetic fraction encoded in
  that value;
- no parsed fraction is automatically labelled a conventional starting price;
- no derived value is automatically treated as comparable across races;
- market type and wagering applicability remain separate classification
  questions;
- ambiguous cases remain unresolved until supported by race context or external
  evidence.

The bounded question is consequently refined to:

> How is the source `sp` field represented, which values can be parsed
> reproducibly, and what limits prevent it from being treated globally as a
> conventional starting-price field?

In [40]:
# Build a race-level coverage classification for the source `sp` field.
#
# This does not assign a market type. It records only the observed distribution
# of blank and nonblank values within each provisional race.

with sqlite3.connect(READ_ONLY_DB_URI, uri=True) as connection:
    race_price_coverage_profile = pd.read_sql_query(
        f"""
        WITH race_rows AS (
            SELECT
                date,
                course,
                off,
                COUNT(*) AS source_runner_records,
                MAX(ran) AS declared_ran,
                SUM(CASE WHEN sp = '' THEN 1 ELSE 0 END) AS blank_rows,
                SUM(CASE WHEN sp <> '' THEN 1 ELSE 0 END) AS nonblank_rows,
                MIN(
                    CASE
                        WHEN sp <> ''
                         AND CAST(pos AS TEXT) GLOB '[1-9]*'
                        THEN CAST(pos AS INTEGER)
                    END
                ) AS minimum_priced_position,
                MAX(
                    CASE
                        WHEN sp <> ''
                         AND CAST(pos AS TEXT) GLOB '[1-9]*'
                        THEN CAST(pos AS INTEGER)
                    END
                ) AS maximum_priced_position,
                COUNT(
                    DISTINCT CASE
                        WHEN sp <> ''
                         AND CAST(pos AS TEXT) GLOB '[1-9]*'
                        THEN CAST(pos AS INTEGER)
                    END
                ) AS distinct_numeric_priced_positions,
                SUM(
                    CASE
                        WHEN sp <> ''
                         AND NOT (CAST(pos AS TEXT) GLOB '[1-9]*')
                        THEN 1 ELSE 0
                    END
                ) AS priced_non_numeric_outcomes
            FROM {SOURCE_TABLE}
            WHERE {DATA_ROW_PREDICATE}
            GROUP BY
                date,
                course,
                off
        ),
        classified AS (
            SELECT
                *,
                CASE
                    WHEN blank_rows = 0
                        THEN 'complete_nonblank_coverage'

                    WHEN nonblank_rows = 0
                        THEN 'all_runners_blank'

                    WHEN nonblank_rows = 1
                     AND minimum_priced_position = 1
                     AND maximum_priced_position = 1
                     AND priced_non_numeric_outcomes = 0
                        THEN 'winner_only_nonblank'

                    WHEN minimum_priced_position = 1
                     AND maximum_priced_position = nonblank_rows
                     AND distinct_numeric_priced_positions = nonblank_rows
                     AND priced_non_numeric_outcomes = 0
                        THEN 'contiguous_leading_finishers_nonblank'

                    ELSE 'irregular_partial_coverage'
                END AS coverage_pattern
            FROM race_rows
        )
        SELECT
            coverage_pattern,
            COUNT(*) AS provisional_races,
            SUM(source_runner_records) AS runner_records,
            SUM(blank_rows) AS blank_runner_records,
            SUM(nonblank_rows) AS nonblank_runner_records,
            SUM(
                CASE
                    WHEN source_runner_records = declared_ran
                    THEN 1 ELSE 0
                END
            ) AS complete_extract_races,
            SUM(
                CASE
                    WHEN source_runner_records < declared_ran
                    THEN 1 ELSE 0
                END
            ) AS incomplete_extract_races
        FROM classified
        GROUP BY coverage_pattern
        ORDER BY provisional_races DESC;
        """,
        connection,
    )

race_price_coverage_profile

,coverage_pattern,provisional_races,runner_records,blank_runner_records,nonblank_runner_records,complete_extract_races,incomplete_extract_races
0,complete_nonblank_coverage,188148,1841428,0,1841428,188147,1
1,all_runners_blank,701,7766,7766,0,697,4
2,winner_only_nonblank,122,1272,1150,122,122,0
3,contiguous_leading_finishers_nonblank,53,607,159,448,53,0
4,irregular_partial_coverage,19,212,22,190,19,0


In [41]:
# Locate the winner-only and leading-finisher coverage patterns by course.
# This will identify the jurisdictions and venues where the `sp` field is most
# likely to contain selective returns rather than conventional all-runner SPs.

with sqlite3.connect(READ_ONLY_DB_URI, uri=True) as connection:
    selective_price_coverage_locations = pd.read_sql_query(
        f"""
        WITH race_rows AS (
            SELECT
                date,
                course,
                off,
                COUNT(*) AS source_runner_records,
                SUM(CASE WHEN sp = '' THEN 1 ELSE 0 END) AS blank_rows,
                SUM(CASE WHEN sp <> '' THEN 1 ELSE 0 END) AS nonblank_rows,
                MIN(
                    CASE
                        WHEN sp <> ''
                         AND CAST(pos AS TEXT) GLOB '[1-9]*'
                        THEN CAST(pos AS INTEGER)
                    END
                ) AS minimum_priced_position,
                MAX(
                    CASE
                        WHEN sp <> ''
                         AND CAST(pos AS TEXT) GLOB '[1-9]*'
                        THEN CAST(pos AS INTEGER)
                    END
                ) AS maximum_priced_position,
                COUNT(
                    DISTINCT CASE
                        WHEN sp <> ''
                         AND CAST(pos AS TEXT) GLOB '[1-9]*'
                        THEN CAST(pos AS INTEGER)
                    END
                ) AS distinct_numeric_priced_positions,
                SUM(
                    CASE
                        WHEN sp <> ''
                         AND NOT (CAST(pos AS TEXT) GLOB '[1-9]*')
                        THEN 1 ELSE 0
                    END
                ) AS priced_non_numeric_outcomes
            FROM {SOURCE_TABLE}
            WHERE {DATA_ROW_PREDICATE}
            GROUP BY
                date,
                course,
                off
        ),
        classified AS (
            SELECT
                *,
                CASE
                    WHEN nonblank_rows = 1
                     AND minimum_priced_position = 1
                     AND maximum_priced_position = 1
                     AND priced_non_numeric_outcomes = 0
                        THEN 'winner_only_nonblank'

                    WHEN blank_rows > 0
                     AND nonblank_rows > 0
                     AND minimum_priced_position = 1
                     AND maximum_priced_position = nonblank_rows
                     AND distinct_numeric_priced_positions = nonblank_rows
                     AND priced_non_numeric_outcomes = 0
                        THEN 'contiguous_leading_finishers_nonblank'

                    ELSE NULL
                END AS coverage_pattern
            FROM race_rows
        )
        SELECT
            coverage_pattern,
            course,
            COUNT(*) AS provisional_races,
            SUM(source_runner_records) AS runner_records,
            SUM(nonblank_rows) AS nonblank_runner_records,
            MIN(date) AS first_date,
            MAX(date) AS last_date
        FROM classified
        WHERE coverage_pattern IS NOT NULL
        GROUP BY
            coverage_pattern,
            course
        ORDER BY
            coverage_pattern,
            provisional_races DESC,
            course;
        """,
        connection,
    )

selective_price_coverage_locations

,coverage_pattern,course,provisional_races,runner_records,nonblank_runner_records,first_date,last_date
0,contiguous_leading_finishers_nonblank,San Isidro (ARG),17,174,85,2015-06-27,2015-12-12
1,contiguous_leading_finishers_nonblank,Auteuil (FR),7,72,59,2016-06-20,2024-03-09
2,contiguous_leading_finishers_nonblank,Kyoto (JPN),3,51,48,2019-10-06,2019-11-24
3,contiguous_leading_finishers_nonblank,Nakayama (JPN),3,40,36,2018-09-30,2019-04-13
4,contiguous_leading_finishers_nonblank,Tokyo (JPN),3,46,43,2017-02-18,2025-05-10
5,contiguous_leading_finishers_nonblank,Happy Valley (HK),2,23,21,2017-11-22,2020-03-11
6,contiguous_leading_finishers_nonblank,Saratoga (USA),2,15,13,2021-07-24,2023-09-03
7,contiguous_leading_finishers_nonblank,Baden-Baden (GER),1,7,6,2021-09-05,2021-09-05
8,contiguous_leading_finishers_nonblank,Caulfield (AUS),1,14,13,2017-10-21,2017-10-21
9,contiguous_leading_finishers_nonblank,Chantilly (FR),1,10,9,2023-12-22,2023-12-22


## Geographical concentration of selective price or return coverage

Selective nonblank `sp` coverage is concentrated by course and jurisdiction.

### Winner-only coverage

The 122 winner-only races are dominated by:

- Monterrico, including both `Monterrico (PER)` and later `Monterrico`
  representations;
- French provincial and regional courses;
- St Moritz;
- a small number of isolated races elsewhere.

Monterrico alone accounts for 66 winner-only races across the two observed
course-name forms.

Several French venues show repeated winner-only coverage across multiple years,
including:

- Deauville;
- Le Touquet;
- Morlaix;
- Senonnes-Pouance;
- Durtal;
- La Roche-Posay;
- Sille-Le-Guillaume.

This concentration supports a systematic source or wagering-return convention
rather than random runner-level missingness.

### Leading-finisher coverage

The 53 races with prices for a contiguous block of leading finishers are led by:

- San Isidro;
- Auteuil;
- Kyoto;
- Nakayama;
- Tokyo;
- Happy Valley;
- Saratoga.

San Isidro accounts for 18 such races across the observed
`San Isidro (ARG)` and `San Isidro` course-name forms.

Some venues provide prices for only the first five finishers, while others
provide prices for nearly the entire field. The source therefore does not use
one universal leading-finisher threshold.

### Retained consequence

Selective `sp` coverage is associated with course, jurisdiction and period.

The later jurisdiction and betting-market study must therefore consider:

- course-name variants across time;
- jurisdiction-specific wagering systems;
- venue-level reporting conventions;
- whether the stored value is a conventional starting price, a tote dividend
  or another return;
- whether nonblank coverage normally applies to all runners, the winner only
  or a limited group of finishers.

Notebook 08 records these observed patterns without assigning a final market
classification to each course.

In [42]:
# Apply the parser to every runner record and compare its results with
# independent SQL counts from the source table.
#
# This checks that the distinct-value parser scales correctly to runner-record
# grain and that no current source value is omitted or miscounted.

with sqlite3.connect(READ_ONLY_DB_URI, uri=True) as connection:
    runner_sp_values = pd.read_sql_query(
        f"""
        SELECT
            rowid AS source_rowid,
            sp AS raw_sp
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
        ORDER BY rowid;
        """,
        connection,
    )

runner_parse_results = pd.DataFrame(
    [
        {
            "source_rowid": source_rowid,
            **parse_starting_price(raw_sp).__dict__,
        }
        for source_rowid, raw_sp in runner_sp_values.itertuples(
            index=False,
            name=None,
        )
    ]
)

runner_parser_validation = pd.DataFrame(
    {
        "measure": [
            "runner records parsed",
            "numeric-price parsed rows",
            "empty-string missing rows",
            "marker-only rows",
            "unsupported current rows",
            "unreduced-fraction rows",
            "standalone numeric-price-absent rows",
            "duplicate source rowids",
            "minimum source rowid",
            "maximum source rowid",
        ],
        "value": [
            len(runner_parse_results),
            int(runner_parse_results["parse_status"].eq("parsed").sum()),
            int(runner_parse_results["parse_status"].eq("missing").sum()),
            int(
                runner_parse_results["parse_status"]
                .eq("price_absent_from_source_string")
                .sum()
            ),
            int(runner_parse_results["parse_status"].eq("unresolved").sum()),
            int(
                runner_parse_results["anomaly_flags"]
                .map(lambda flags: "fraction_not_in_lowest_terms" in flags)
                .sum()
            ),
            int(
                runner_parse_results["anomaly_flags"]
                .map(lambda flags: "numeric_price_absent" in flags)
                .sum()
            ),
            int(runner_parse_results["source_rowid"].duplicated().sum()),
            int(runner_parse_results["source_rowid"].min()),
            int(runner_parse_results["source_rowid"].max()),
        ],
    }
)

runner_parser_validation

,measure,value
0,runner records parsed,1851285
1,numeric-price parsed rows,1842187
2,empty-string missing rows,9097
3,marker-only rows,1
4,unsupported current rows,0
5,unreduced-fraction rows,77468
6,standalone numeric-price-absent rows,1
7,duplicate source rowids,0
8,minimum source rowid,2
9,maximum source rowid,1851286


In [43]:
# Run failure-detecting notebook assertions against the current source domain.
#
# These assertions should fail if:
# - source row counts change unexpectedly;
# - a new unsupported `sp` structure appears;
# - the known missing or marker-only counts change;
# - parser coverage loses source rows;
# - physical source lineage is duplicated.

assert len(runner_parse_results) == 1_851_285

assert runner_parse_results["source_rowid"].is_unique
assert runner_parse_results["source_rowid"].min() == 2
assert runner_parse_results["source_rowid"].max() == 1_851_286

assert runner_parse_results["parse_status"].eq("parsed").sum() == 1_842_187
assert runner_parse_results["parse_status"].eq("missing").sum() == 9_097

assert (
    runner_parse_results["parse_status"]
    .eq("price_absent_from_source_string")
    .sum()
    == 1
)

assert runner_parse_results["parse_status"].eq("unresolved").sum() == 0

assert (
    runner_parse_results["anomaly_flags"]
    .map(lambda flags: "fraction_not_in_lowest_terms" in flags)
    .sum()
    == 77_468
)

assert (
    runner_parse_results["anomaly_flags"]
    .map(lambda flags: "numeric_price_absent" in flags)
    .sum()
    == 1
)

print("Notebook starting-price parser validation passed.")

Notebook starting-price parser validation passed.


## Notebook conclusion

### Bounded answer

The source column is named `sp`, but it cannot be interpreted globally as a
conventional runner-level starting-price field.

It is a mixed source price-and-return field whose meaning varies by race,
course, jurisdiction and wagering context.

### Current source representation

Across all 1,851,285 data-like runner records:

- every `sp` value is stored as SQLite text;
- there are 843 distinct raw values;
- 9,097 records contain an empty string;
- there are no SQL `NULL` values;
- there are no whitespace-only values;
- there are no values with outer whitespace.

The current nonblank notation consists of:

- fractional expressions;
- textual even-money expressions:
  - `Evs`;
  - `EvensF`;
  - `EvsJ`;
- terminal favourite markers:
  - `F`;
  - `J`;
  - `C`;
- one standalone `F` value without a numeric component.

No current value uses decimal-odds notation or another unsupported structure.

### Reproducible parsing rule

The current source supports deterministic parsing of:

```text
<numerator>/<denominator>[optional marker]
```

where:

- numerator and denominator are positive integers;
- the optional terminal marker is `F`, `J` or `C`;
- fractions are not required to be in lowest terms.

The textual even-money values map arithmetically to `1/1`, while preserving
their exact raw notation and any favourite marker.

The parser preserves separately:

- exact raw `sp`;
- notation family;
- raw numerator;
- raw denominator;
- favourite marker;
- parse status;
- anomaly flags.

### Parser coverage

The final notebook parser processed all 1,851,285 data-like runner records:

- 1,842,187 records contain a reproducibly parsed numeric fraction;
- 9,097 records contain an explicit empty string;
- one record contains a standalone favourite marker with no numeric component;
- zero current records contain an unsupported structure;
- 77,468 records contain valid fractions that are not in lowest terms.

The standalone `F` belongs to Almendares in the 2025 Wickerr Stakes at Del Mar.
Its source string contains only favourite status, but the missing numeric price
has been externally verified as `5/2 favourite`. That external correction must
remain separate from parser-derived output.

### Favourite markers

Observed race-level behaviour strongly supports:

- `F` — favourite;
- `J` — joint favourite;
- `C` — co-favourite.

However, marker counts are not fully consistent across races or jurisdictions.
The parser therefore preserves the literal marker without enforcing a universal
race-level count rule.

### Unreduced and precision-heavy fractions

The source does not consistently reduce fractional expressions.

Examples include:

```text
100/30
52/10
145/20
30/100F
```

The exact stored numerator and denominator are source evidence and must not be
overwritten by a reduced form.

Precision-heavy denominators occur throughout the 2015–2026 source period and
across domestic and international courses. External checks confirm that at
least some such values are arithmetically valid representations of published
returns.

Their original upstream notation and conversion method are not documented by
the source.

### Mixed field semantics

Race-context and external inspection established that `sp` may contain:

- a conventional fixed-odds starting price;
- a fractional representation of a pari-mutuel or tote dividend;
- a return supplied only for the winner;
- returns supplied for a limited group of leading finishers;
- a favourite marker without a numeric source value;
- no value because the runner was excluded from wagering;
- no value because the race supplied no usable price or return;
- an unexplained source omission.

Therefore, a parsed numeric fraction establishes only the arithmetic meaning of
the stored expression. It does not by itself establish:

- the original market type;
- the native jurisdictional notation;
- whether the value is a bookmaker SP or tote dividend;
- whether it applied to every runner;
- whether prices are directly comparable across jurisdictions.

### Race-level coverage patterns

Across 189,043 provisional races:

- 188,148 have complete nonblank coverage;
- 701 have all runners blank;
- 122 have a nonblank value for the winner only;
- 53 have nonblank values for a contiguous group of leading finishers;
- 19 have irregular partial coverage.

These patterns occur overwhelmingly in complete race extracts and are
geographically concentrated, so they represent genuine source behaviour rather
than missing physical runner rows.

Blank `sp` values must therefore remain associated with race-level coverage
context and must not be collapsed into one undifferentiated missing category.

### Known exceptions and preservation mechanism

The parser deliberately leaves unresolved any future value containing:

- SQL `NULL`;
- non-text storage;
- outer whitespace;
- leading-zero components;
- zero numerator or denominator;
- decimal notation;
- metric notation;
- unknown suffixes;
- multiple suffixes;
- unfamiliar alphabetic tokens;
- any unsupported future structure.

Such values must be:

1. preserved exactly;
2. surfaced by independent validation;
3. reviewed in race context;
4. checked externally where necessary;
5. added to the parser only when a defensible general rule is established.

### Database consequence

Later processing must preserve:

- exact raw `sp`;
- parsed arithmetic components;
- literal favourite marker;
- parser status and anomaly flags;
- race-level price-coverage pattern;
- externally verified corrections as separate audit records;
- evidence about wagering applicability and market type where available.

The final schema remains deferred.

### Deferred prerequisite study

A later notebook must study:

```text
Course Jurisdiction, Racing Authority and Betting-Market Context
```

That work will be required before distance, carried weight or `sp`-derived
values are treated as directly comparable across jurisdictions.

### Stopping rule satisfied

Notebook 08 now has:

- a reproducible parsing rule for every supported current value;
- documented missing, marker-only and mixed-semantic exceptions;
- preservation rules for raw and externally corrected values;
- failure-detecting notebook assertions;
- zero unsupported current structures;
- a bounded conclusion about what the source field does and does not justify.

The analytical notebook can now proceed to **Restart Kernel and Run All Cells**.